# IAA (Krippendorff's $\alpha$)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns

from krippendorff import alpha
from numpy import dtype, ndarray
from tqdm import tqdm
from typing import Any, Literal

INPUT_DATA_PATH = os.path.join("../data", "students_teacher_gold.json")

In [ ]:
def get_data(input_data_path: str = INPUT_DATA_PATH) -> list[list[dict[str, Any]]]:
    with open(input_data_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return [item.get("teacher_annotation_data", []) for item in data]

def get_annotators(input_data_path: str = INPUT_DATA_PATH) -> list[str]:
    data = get_data(input_data_path)
    
    annotators = set()
    for annotations in data:
        for annotation in annotations:
            id = annotation.get("annotator_id")
            if id is not None:
                annotators.add(id)

    return sorted(list(annotators))

def get_n_annotated_items(input_data_path: str = INPUT_DATA_PATH) -> int:
    return len(get_data(input_data_path))

def load_data_matrix(input_path: str = INPUT_DATA_PATH) -> ndarray[tuple[int, int], dtype[Any]]:
    """
    Load the gold JSON data and convert it into a matrix, where each row corresponds to an annotator and each column corresponds to an annotated item. The matrix contains the numeric representation of the annotations (1 for model A, -1 for model B, 0 for ties), and fills missing annotations with `NaN` values.
    The output matrix can be used to compute Krippendorff's alpha.
    """
    data = get_data(input_path)
    
    n_annotated_items = get_n_annotated_items(input_path)
    annotators = get_annotators(input_path)
    
    annotator_to_idx = {annotator: idx for idx, annotator in enumerate(annotators)}
    
    map_annotation_to_numeric = {
        "a": 1,
        "both_equal": 0,
        "b": -1
    }

    out_matrix = np.full((len(annotators), n_annotated_items), np.nan)
    
    for i, item in enumerate(data):
        for annotation in item:
            annotator_id = annotation.get("annotator_id")
            label = map_annotation_to_numeric.get(annotation.get("preferred_model", ""), None)
            
            if annotator_id is not None and annotator_id in annotator_to_idx and label is not None:
                out_matrix[annotator_to_idx[annotator_id], i] = label
    
    return out_matrix

In [ ]:
# def compute_bootstrapped_alpha(data_matrix: np.ndarray, level_of_measurement: Literal['nominal', 'ordinal'] = 'nominal', value_domain: list = None, n_iterations: int = 20000, p_value: float = 0.05, seed: int | None = 2026) -> tuple[np.floating, np.float64, np.float64]:
#     """
#     Computes Krippendorff's alpha confidence intervals using the 
#     bootstrapping algorithm defined by K. Krippendorff (2006):
#     *"Bootstrapping Distributions for Krippendorff’s Alpha
#     for coding predefined units: single-valued c_alpha and multi-valued mv_alpha"*
    
#     Source: https://www.asc.upenn.edu/sites/default/files/2021-03/Algorithm%20for%20Bootstrapping%20a%20Distribution%20of%20Alpha.pdf
#     (Backup Archive:) https://web.archive.org/web/20260209232738/https://www.asc.upenn.edu/sites/default/files/2021-03/Algorithm%20for%20Bootstrapping%20a%20Distribution%20of%20Alpha.pdf
    
#     Based on the original metric: https://www.asc.upenn.edu/sites/default/files/2021-03/Computing%20Krippendorff's%20Alpha-Reliability.pdf
#     (Backup Archive:) https://web.archive.org/web/20260624173717/https://www.asc.upenn.edu/sites/default/files/2021-03/Computing%20Krippendorff's%20Alpha-Reliability.pdf
    
#     Returns:
    
#     A 3-tuple (observed alpha, lower bound of confidence interval, upper bound of confidence interval).
#     """
#     rng = np.random.default_rng(seed)
    
#     n_observers, n_items = data_matrix.shape

#     # -- Data preparation and parameters
    
#     # "Number of pairable values m_u"
#     m_u_list = np.sum(~np.isnan(data_matrix), axis=0)

#     # "The number N_o of unique pairs that contribute to alpha"
#     N_o = int(np.sum((m_u_list - 1) * m_u_list / 2))

#     # "n.. = sum(m_u)" (Total number of pairable values)
#     n_dotdot = np.sum(m_u_list)

#     if N_o == 0 or n_dotdot == 0:
#         raise ValueError("Not enough pairable data to compute alpha.")

#     # Calculate expected disagreement (D_e) from the observed data.
#     # We maintain "the generally more stable expected disagreement D_e from the observed data."
#     valid_values = data_matrix[~np.isnan(data_matrix)]
    
#     if value_domain is not None and type(value_domain) is list and len(value_domain) > 0:
#         # If a domain is provided, we use it to ensure that all categories are included in the counts, even if they have zero occurrences
#         unique_vals = np.array(value_domain)
#         # We order the unique values to ensure consistent ordering
#         unique_vals = np.sort(unique_vals) 
#         # We count the frequencies by forcing the inclusion of empty categories (0 occurrences)
#         counts = np.array([np.sum(valid_values == val) for val in unique_vals])
#     else:
#         # Otherwise, we compute the unique values and their counts from the observed data
#         unique_vals, counts = np.unique(valid_values, return_counts=True)
#         sort_idx = np.argsort(unique_vals)
#         unique_vals = unique_vals[sort_idx]
#         counts = counts[sort_idx]
    
#     K = len(unique_vals)
#     val_to_idx = {val: idx for idx, val in enumerate(unique_vals)}
    
#     # value_probs = counts / n_dotdot
    
#     # D_e = 1.0 - np.sum(value_probs ** 2) # Nominal metric D_e
    
#     # if D_e == 0:
#     #     raise ValueError("Reliability data have no variance whereby alpha = 1 - 0/0 = 0 by definition.")

#     # -- Parameterized distance matrix (delta^2)
    
#     delta_sq = np.zeros((K, K))
    
#     for i in range(K):
#         for j in range(K):
#             if level_of_measurement == 'nominal':
#                 # Nominal distance: 0 if identical, 1 if different
#                 delta_sq[i, j] = 0.0 if i == j else 1.0
#             elif level_of_measurement == 'ordinal':
#                 # Ordinal distance: relies on cumulative rank frequencies
#                 if i == j:
#                     delta_sq[i, j] = 0.0
#                 else:
#                     c, k = min(i, j), max(i, j)
#                     sum_counts = np.sum(counts[c:k + 1])
#                     dist = (sum_counts - counts[c] / 2.0 - counts[k] / 2.0)
#                     delta_sq[i, j] = dist ** 2
#             else:
#                 raise ValueError(f"Unsupported level of measurement: {level_of_measurement}")
    
#     # -- Universal expected disagreement (D_e)
    
#     D_e = 0.0
#     for i in range(K):
#         for j in range(K):
#             D_e += counts[i] * counts[j] * delta_sq[i, j]
#     D_e /= n_dotdot * (n_dotdot - 1)
    
#     if D_e == 0:
#         raise ValueError("Reliability data have no variance (D_e = 0).")

#     # -- Deviations array creation
    
#     E_list = []
#     unit_slices: list[tuple[int, int]] = [] # (start, end) per item with m_u >= 2
    
#     # "Create a list of N_o items"
#     for u in range(n_items):
#         unit_data = data_matrix[:, u]
#         unit_vals = unit_data[~np.isnan(unit_data)]
#         m_u = len(unit_vals)

#         start = len(E_list)

#         if m_u >= 2:
#             for i in range(m_u):
#                 for j in range(i + 1, m_u):
#                     # "metric delta^2" (For nominal: 0 if c == k, else 1)
#                     idx1 = val_to_idx[unit_vals[i]]
#                     idx2 = val_to_idx[unit_vals[j]]
                    
#                     d_sq = delta_sq[idx1, idx2]
                    
#                     # "The r-th out of N_o possible deviations E(r) from alpha=1 is:"
#                     # E(r) = 2 * (metric delta^2) / (n.. * D_e)
#                     E_r = 2.0 * d_sq / (n_dotdot * D_e)
#                     E_list.append(E_r)
#         end = len(E_list)
#         unit_slices.append((start, end))

#     E_array = np.array(E_list)
    
#     assert len(E_array) == N_o, f"Expected {N_o} deviations, but got {len(E_array)}."

#     # -- Deterministic alpha computation (observed alpha)
    
#     alpha_observed = 1.0
#     for u, (start, end) in enumerate(unit_slices):
#         m_u = m_u_list[u]
#         if m_u >= 2:
#             alpha_observed -= E_array[start:end].sum() / (m_u - 1)

#     # -- Bootstrapping algorithm

#     pair_counts = np.array([end - start for start, end in unit_slices])
#     total_pairs = pair_counts.sum()
    
#     # One big draw instead of drawing one by one to improve performance
#     all_r = rng.integers(0, N_o, size=(n_iterations, total_pairs))
#     all_E = E_array[all_r]

#     # # "Set the integer array n_alpha = 0"
#     # # Note: We use a raw array to store exact alphas for precise percentile computation
#     # bootstrapped_alphas = np.zeros(n_iterations)

#     # # "Do X times" (X = 20,000 suggested)
#     # for x in range(n_iterations):
        
#     #     # "alpha = 1"
#     #     alpha = 1.0

#     #     # "Do u = 1, N_u"
#     #     for u in range(n_items):
#     #         m_u = m_u_list[u]
#     #         if m_u >= 2:
#     #             n_pairs_in_u = int((m_u - 1) * m_u / 2)

#     #             # "Do (m_u - 1)m_u / 2 times (= The number of unique pairs in the u-th unit)"
#     #             for _ in range(n_pairs_in_u):
                    
#     #                 # "Pick a random integer 1 <= r <= N_o (uniform distribution)"
#     #                 # Note: Python is 0-indexed, so 0 <= r < N_o
#     #                 r = rng.integers(0, N_o)

#     #                 # "alpha <- alpha - E(r) / (m_u - 1)"
#     #                 alpha -= E_array[r] / (m_u - 1)

#     #     # "If alpha < -1: n_-1 = n_-1 + 1"
#     #     if alpha < -1.0:
#     #         alpha = -1.0

#     #     bootstrapped_alphas[x] = alpha

#     bootstrapped_alphas = np.full(n_iterations, 1.0)
#     col = 0
#     for u, (start, end) in enumerate(unit_slices):
#         m_u = m_u_list[u]
#         n_pairs_in_u = end - start
#         if m_u >= 2:
#             chunk = all_E[:, col:col + n_pairs_in_u].sum(axis=1)
#             bootstrapped_alphas -= chunk / (m_u - 1)
#         col += n_pairs_in_u
        
#     bootstrapped_alphas = np.clip(bootstrapped_alphas, -1.0, None)  # Ensure alpha >= -1

#     # -- Extracting confidence intervals
    
#     # "The confidence interval: -1 <= alpha_smallest <= alpha <= alpha_largest <= 1"
#     # "for a chosen level p of statistical significance (two-tailed)"
    
#     # "min >= 1 - p"
#     # "max <= p / 2"
#     lower_percentile = (p_value / 2.0) * 100
#     upper_percentile = (1.0 - p_value / 2.0) * 100

#     alpha_smallest = np.percentile(bootstrapped_alphas, lower_percentile)
#     alpha_largest = np.percentile(bootstrapped_alphas, upper_percentile)

#     print(f"Observed Alpha : {alpha_observed:.3f}")
#     print(f"Confidence Interval : [{alpha_smallest:.3f}, {alpha_largest:.3f}]")

#     return alpha_observed, alpha_smallest, alpha_largest

In [ ]:
# print("Krippendorff's alpha:", alpha(load_data_matrix(INPUT_DATA_PATH), value_domain=[-1, 0, 1], level_of_measurement="ordinal"))
# compute_bootstrapped_alpha(load_data_matrix(INPUT_DATA_PATH), level_of_measurement="ordinal", value_domain=[-1, 0, 1], n_iterations=20000, p_value=0.05, seed=2026)

In [ ]:
def compute_alpha_with_ci(
    data_matrix: np.ndarray,
    level_of_measurement: Literal['nominal', 'ordinal'] = 'nominal', 
    value_domain: list = None,
    n_iterations: int = 2000,
    p_value: float = 0.05,
    seed: int = 2026
) -> tuple[float, float, float]:
    """
    Computes Krippendorff's alpha and its confidence intervals using non-parametric bootstrapping on the units (columns) of the data matrix.
    """
    rng = np.random.default_rng(seed)
    n_annotators, n_items = data_matrix.shape

    # -- Computing the observed alpha
    try:
        alpha_observed = alpha(
            reliability_data=data_matrix,
            level_of_measurement=level_of_measurement,
            value_domain=value_domain
        )
    except ValueError as e:
        raise ValueError(f"Could not compute observed alpha: {e}")

    bootstrapped_alphas = []

    # Vectorized generation of indices to speed up the loop
    # Matrix of (n_iterations, n_items) containing the randomly drawn indices
    all_resampled_indices = rng.choice(n_items, size=(n_iterations, n_items), replace=True)

    for i in tqdm(range(n_iterations), desc="Bootstrapping"):
        indices = all_resampled_indices[i]
        resampled_matrix = data_matrix[:, indices]

        try:
            a = alpha(
                reliability_data=resampled_matrix,
                level_of_measurement=level_of_measurement,
                value_domain=value_domain
            )
            bootstrapped_alphas.append(a)
        except ValueError:
            # If the bootstrap fails for this iteration (e.g., due to lack of variance), we skip it
            continue

    bootstrapped_alphas = np.array(bootstrapped_alphas)
    bootstrapped_alphas = bootstrapped_alphas[~np.isnan(bootstrapped_alphas)]

    if len(bootstrapped_alphas) == 0:
        raise ValueError("Bootstrap failed: null variance on all samples.")

    lower_percentile = (p_value / 2.0) * 100
    upper_percentile = (1.0 - p_value / 2.0) * 100

    alpha_smallest = np.percentile(bootstrapped_alphas, lower_percentile)
    alpha_largest = np.percentile(bootstrapped_alphas, upper_percentile)
    mean_bootstrap = np.mean(bootstrapped_alphas)

    print("-" * 60)
    print(f"Used Metric: {level_of_measurement}".center(60))
    print(f"True Observed Alpha: {alpha_observed:.3f}".center(60))
    print(f"Mean of Bootstrap: {mean_bootstrap:.3f} (Used as a bias diagnostic)".center(60))
    print(f"Confidence Interval at {int((1-p_value)*100)}%: [{alpha_smallest:.3f}, {alpha_largest:.3f}]".center(60))
    print("-" * 60)

    return float(alpha_observed), float(alpha_smallest), float(alpha_largest)

In [ ]:
compute_alpha_with_ci(
    data_matrix=load_data_matrix(INPUT_DATA_PATH),
    level_of_measurement="ordinal",
    value_domain=[-1, 0, 1],
    n_iterations=20000,
    p_value=0.05,
    seed=2026
)

In [ ]:
def visualize_matrix(matrix: np.ndarray, cmap: sns.palettes._ColorPalette, ticks: list[int], ticks_label: str, title: str, xlabel: str, ylabel: str) -> None:
    plt.figure(figsize=(16, 6))
    
    ax = sns.heatmap(
        matrix,
        cmap=cmap,
        cbar_kws={'ticks': ticks, 'label': ticks_label},
        linewidths=0.05,
        linecolor='whitesmoke'
    )
    
    ax.set_facecolor('#f0f0f0')
    
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    
    plt.tight_layout()
    plt.show()

visualize_matrix(matrix=load_data_matrix(), cmap = sns.color_palette(["#e74c3c", "#95a5a6", "#2ecc71"]), ticks=[-1, 0, 1], ticks_label='Preference (B / Tie / A)', title='Annotation Matrix Visualization', xlabel='Item indices', ylabel='Annotator indices')

## Rating-wise

In [ ]:
def load_matrix_ratings(input_path: str = INPUT_DATA_PATH) -> ndarray[tuple[int, int], dtype[Any]]:
    """
    Load annotation matrix from the gold JSON dataset, where each row corresponds to an annotator and each column corresponds to an annotated item. The matrix contains the numeric representation of the ratings (1-5), and fills missing annotations with `NaN` values. Only items with a consensus among all annotators (exactly 1 unique model chosen by all) are included in the matrix. The output matrix can be used to compute Krippendorff's alpha.
    """
    data = get_data(input_path)
    annotators = get_annotators(input_path)
    annotator_to_idx = {annotator: idx for idx, annotator in enumerate(annotators)}
    
    # Matrix filled with NaN values by default
    out_matrix = np.full((len(annotators), len(data)), np.nan)
    
    for i, item in enumerate(data):
        
        # Get all preferred models for this conversation
        preferred_models = set(
            ann.get("preferred_model")
            for ann in item
            if ann.get("preferred_model") is not None
        )
        
        # If there isn't an absolute consensus among all annotators (exactly 1 unique model chosen by all), we ignore this item and move on to the next one
        if len(preferred_models) != 1:
            continue
        
        for annotation in item:
            annotator_id = annotation.get("annotator_id")
            label: Literal[0, 1, 2, 3, 4, 5] = annotation.get("rating", 0)
            
            if annotator_id is not None and annotator_id in annotator_to_idx and label != 0: # 0 means no label provided
                out_matrix[annotator_to_idx[annotator_id], i] = label
                
    return out_matrix

compute_alpha_with_ci(
    data_matrix=load_matrix_ratings(),
    level_of_measurement="ordinal",
    value_domain=[1, 2, 3, 4, 5],
    n_iterations=20000,
    p_value=0.05,
    seed=2026
)

visualize_matrix(matrix=load_matrix_ratings(), cmap = sns.color_palette(["#ff0000", "#888888", "#00ff00", "#228822"]), ticks=[1, 2, 3, 4, 5], ticks_label='Rating (1-5)', title='Rating-based Annotation Matrix Visualization', xlabel='Item indices', ylabel='Annotator indices')


## Label-wise

In [ ]:
LABELS = ['complete', 'correct', 'relevant', 'concise', 'scaffolding', 'understandable']

def load_matrix_labels(target_label: Literal['complete', 'correct', 'relevant', 'concise', 'scaffolding', 'understandable'], input_path: str = INPUT_DATA_PATH, ) -> ndarray[tuple[int, int], dtype[Any]]:
    """
    Load annotation matrix from the gold JSON dataset, where each row corresponds to an annotator and each column corresponds to an annotated item. The matrix contains the boolean representation of a label (whether chosen or not), and fills missing annotations with `NaN` values. Only items with a consensus among all annotators (exactly 1 unique model chosen by all) are included in the matrix. The output matrix can be used to compute Krippendorff's alpha.
    """
    data = get_data(input_path)
    annotators = get_annotators(input_path)
    annotator_to_idx = {annotator: idx for idx, annotator in enumerate(annotators)}
    
    # Matrix filled with NaN values by default
    out_matrix = np.full((len(annotators), len(data)), np.nan)
    
    for i, item in enumerate(data):
        
        # Get all preferred models for this conversation
        preferred_models = set(
            ann.get("preferred_model")
            for ann in item
            if ann.get("preferred_model") is not None
        )
        
        # If there isn't an absolute consensus among all annotators (exactly 1 unique model chosen by all), we ignore this item and move on to the next one
        if len(preferred_models) != 1:
            continue
            
        for annotation in item:
            annotator_id = annotation.get("annotator_id")
            label_value = annotation.get(target_label)
            
            if annotator_id is not None and label_value is not None and any([annotation.get(label, False) for label in LABELS]): # We only keep annotations that have at least one label chosen (True) among the six labels => otherwise we assume that the annotator didn't provide any label for this item
                # Boolean values (False/True) are converted to 0 or 1
                out_matrix[annotator_to_idx[annotator_id], i] = int(label_value)
                
    return out_matrix

for label in LABELS:
    print(f" {label} ".center(60, '='))
    try:
        compute_alpha_with_ci(
            data_matrix=load_matrix_labels(target_label=label),
            level_of_measurement="nominal",
            value_domain=[0, 1],
            n_iterations=20000,
            p_value=0.05,
            seed=2026
        )
    except ValueError as e:
        print(f"Could not compute alpha for label '{label}': {e}")
    print('\n')

    visualize_matrix(matrix=load_matrix_labels(target_label=label), cmap = sns.color_palette(["#ff0000", "#228822"]), ticks=[0, 1], ticks_label='Label (No/Yes)', title=f'Label-based Annotation Matrix Visualization ("{label}")', xlabel='Item indices', ylabel='Annotator indices')


# Bradley-Terry-Davidson model

In [ ]:
def get_annotations_list(input_data_path: str = INPUT_DATA_PATH) -> list[tuple[str, str, str]]:
    """Extract annotation data from a JSON file and returns a formatted list of tuples (model_a, model_b, choice) for each annotation, which can be used to compute the Bradley-Terry-Davidson (BTD) model."""
    with open(input_data_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    out = []
    
    for item in data:
        model_a = item.get("model_a_name", "")
        model_b = item.get("model_b_name", "")
        
        for annotation in item.get("teacher_annotation_data", []):
            choice = annotation.get("preferred_model", "")
            if choice in ['a', 'b', 'both_equal']:
                out.append((model_a, model_b, choice))
            else:
                raise ValueError(f"Invalid choice '{choice}' for models '{model_a}' and '{model_b}'. Expected 'a', 'b', or 'both_equal'.")
    
    return out

def compute_btd_from_annotations(annotations_list):
    """
    Computes latent ability scores and the tie parameter using the 
    exact Bradley-Terry-Davidson (BTD) model via Maximum Likelihood Estimation (MLE).
    
    Args:
        annotations_list: List of tuples/lists (model_a, model_b, choice)
                          where choice indicates 'a', 'b', or 'both_equal' (tie).
                          
    Returns:
        results: Dictionary mapping each model name to its BTD latent score and rank.
        final_nu: The estimated Davidson tie parameter (nu > 0).
    """
    
    # Map models to sequential integer indices (alphabetically sorted)
    unique_models = sorted(list(set([m for row in annotations_list for m in row[:2]])))
    n_models = len(unique_models)
    
    model_to_idx = {model: idx for idx, model in enumerate(unique_models)}
    idx_to_model = {idx: model for model, idx in model_to_idx.items()}
    
    # Build aggregate wins and ties matrices
    wins = np.zeros((n_models, n_models), dtype=int)
    ties = np.zeros((n_models, n_models), dtype=int)
    
    for model_a, model_b, choice in annotations_list:
        idx_a = model_to_idx[model_a]
        idx_b = model_to_idx[model_b]
        
        choice_str = str(choice).strip().lower()
        
        # Accumulate wins directionally
        if choice_str == 'a':
            wins[idx_a, idx_b] += 1
        elif choice_str == 'b':
            wins[idx_b, idx_a] += 1
        # Accumulate ties symmetrically
        elif choice_str == 'both_equal':
            ties[idx_a, idx_b] += 1
            ties[idx_b, idx_a] += 1
        else:
            raise ValueError(f"Invalid choice '{choice}' for models '{model_a}' and '{model_b}'. Expected 'a', 'b', or 'both_equal'.")

    # Define the Davidson Negative Log-Likelihood objective function
    def btd_nll(params):
        # params contains: [theta_0, theta_1, ..., theta_{N-2}, nu]
        # We fix the last model's theta to 0.0 for statistical identifiability
        theta = np.append(params[:-1], 0.0)
        nu = params[-1]
        
        # The tie parameter nu must remain strictly positive
        if nu <= 0:
            return np.inf

        nll = 0.0
        
        # Iterate over all unique pairs (i < j)
        for i in range(n_models):
            for j in range(i + 1, n_models):
                w_ij = wins[i, j]
                w_ji = wins[j, i]
                t_ij = ties[i, j] # Symmetrically stored
                
                # Skip unobserved pairs
                if w_ij == 0 and w_ji == 0 and t_ij == 0:
                    continue

                # Exponentiate log-strengths for numerical stability
                exp_i = np.exp(theta[i])
                exp_j = np.exp(theta[j])
                
                # Davidson's joint denominator incorporating the tie parameter nu
                denom = exp_i + exp_j + nu * np.sqrt(exp_i * exp_j)
                
                if denom <= 0:
                    return np.inf

                # Compute log-probabilities for each outcome
                log_p_ij = np.log(exp_i / denom)
                log_p_ji = np.log(exp_j / denom)
                log_p_tie = np.log((nu * np.sqrt(exp_i * exp_j)) / denom)

                # Accumulate the Negative Log-Likelihood (weighted by observed frequencies)
                nll -= (w_ij * log_p_ij + w_ji * log_p_ji + t_ij * log_p_tie)
                
        return nll

    # Optimization via Bounded Maximum Likelihood estimation
    # Initial guesses: all latent scores set to 0.0, nu set to 1.0 (neutral baseline)
    initial_params = np.zeros(n_models)
    initial_params[-1] = 1.0
    
    # Bounds: latent scores can be unconstrained, nu must be > 0 (e.g., 1e-5)
    bounds = [(None, None)] * (n_models - 1) + [(1e-5, None)]

    # Minimize Negative Log-Likelihood using L-BFGS-B optimization algorithm
    with np.errstate(over='ignore', divide='ignore', invalid='ignore'): # Silently ignore cases when one models wins 100% of the time, leading to division by zero
        result = minimize(btd_nll, initial_params, method='L-BFGS-B', bounds=bounds)

    if not result.success:
        raise RuntimeError(f"BTD optimization failed to converge: {result.message}")

    # Post-processing
    final_theta = np.append(result.x[:-1], 0.0)
    
    # Center scores so their mean equals 0.0 for interpretation
    final_theta = final_theta - np.mean(final_theta) 
    final_nu = result.x[-1]

    # Sort models by descending latent score to build the final ranking structure
    sorted_indices = np.argsort(final_theta)[::-1]
    results = {
        idx_to_model[idx]: {
            "score": float(final_theta[idx]),
            "rank": rank + 1,
            "wins/draws/losses": (int(np.sum(wins[idx, :])), int(np.sum(wins[:, idx])), int(np.sum(ties[idx, :])))
        }
        for rank, idx in enumerate(sorted_indices)
    }
    
    return results, final_nu


def compute_btd_with_ci(annotations_list: list, n_iterations: int = 1000, p_value: float = 0.05, seed: int = 2026) -> tuple[dict[Any, dict[str, Any]], Any]:
    """
    Compute the BTD model using bootstrapping to estimate confidence intervals.
    """
    rng = np.random.default_rng(seed)
    
    true_results, true_nu = compute_btd_from_annotations(annotations_list)
    
    models = list(true_results.keys())
    bootstrapped_scores = {model: [] for model in models}
    n_samples = len(annotations_list)
    
    for _ in tqdm(range(n_iterations), desc="Bootstrapping BTD"):
        # Draw a sample of the annotations with replacement
        indices = rng.choice(n_samples, size=n_samples, replace=True)
        resampled_annotations = [annotations_list[i] for i in indices]
        
        try:
            # Recompute the BTD model on the resampled annotations
            res, _ = compute_btd_from_annotations(resampled_annotations)
            
            for model in models:
                if model in res:
                    bootstrapped_scores[model].append(res[model]["score"])
                else:
                    bootstrapped_scores[model].append(np.nan)
                    
        except RuntimeError:
            continue

    lower_percentile = (p_value / 2.0) * 100
    upper_percentile = (1.0 - p_value / 2.0) * 100
    
    for model in models:
        scores_array = np.array(bootstrapped_scores[model])
        
        if len(scores_array) == 0:
            true_results[model]["ci_lower"] = np.nan
            true_results[model]["ci_upper"] = np.nan
            continue
            
        ci_lower = np.nanpercentile(scores_array, lower_percentile)
        ci_upper = np.nanpercentile(scores_array, upper_percentile)
        
        true_results[model]["ci_lower"] = float(ci_lower)
        true_results[model]["ci_upper"] = float(ci_upper)
        true_results[model]["bootstrapped_scores_array"] = scores_array.tolist()
        
    return true_results, true_nu

In [ ]:
btd_scores, nu = compute_btd_with_ci(get_annotations_list(INPUT_DATA_PATH))
print(btd_scores)

In [ ]:
def visualize_btd_scores(scores_dict: dict):
    plt.figure(figsize=(12, 6))
    models = list(scores_dict.keys())
    scores = [scores_dict[model]["score"] for model in models]
    ci_lowers = [scores_dict[model]["ci_lower"] for model in models]
    ci_uppers = [scores_dict[model]["ci_upper"] for model in models]
    
    plt.bar(models, scores, yerr=[np.array(scores) - np.array(ci_lowers), np.array(ci_uppers) - np.array(scores)], capsize=5, color='skyblue', edgecolor='black')
    plt.axhline(0, color='gray', linestyle='--')
    plt.title("Bradley-Terry-Davidson Scores per Model")
    plt.ylabel("Score")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

N_ITERATIONS = 1000

def compute_significance_matrix(scores: dict, n_iterations: int = N_ITERATIONS) -> pd.DataFrame:
    """
    Compute a significance matrix for pairwise comparisons of models based on their bootstrapped scores. The matrix contains empirical p-values and significance stars for each pair of models, indicating whether one model is significantly better than another.
    """
    models = list(scores.keys())
    # Ordering models by their median bootstrapped score for better visualization
    models.sort(key=lambda m: np.median(scores[m]["bootstrapped_scores_array"]), reverse=True)
    
    p_value_matrix = pd.DataFrame(index=models, columns=models, dtype=object)
    stars_matrix = p_value_matrix.copy()
    
    for i, model_a in enumerate(models):
        scores_a = np.array(scores[model_a]["bootstrapped_scores_array"])
        
        for j, model_b in enumerate(models):
            if i == j:
                p_value_matrix.loc[model_a, model_b] = "-"
                stars_matrix.loc[model_a, model_b] = "-"
                continue
                
            scores_b = np.array(scores[model_b]["bootstrapped_scores_array"])
            
            # Empirical p-value: proportion of bootstrap samples where Model B's score >= Model A's score,
            # when A is assumed to be better according to the ranking
            p_value = np.sum(scores_b >= scores_a) / n_iterations
            
            # We keep only the valid scores (where neither is NaN, i.e. both models have scores for a given iteration) to ensure accurate p-value computation
            valid_mask = ~np.isnan(scores_a) & ~np.isnan(scores_b)
            valid_a = scores_a[valid_mask]
            valid_b = scores_b[valid_mask]
            
            if len(valid_a) == 0:
                p_value = np.nan
            else:
                p_value = np.sum(valid_b >= valid_a) / len(valid_a)
            
            if p_value <= 0.001:
                stars = "***"
            elif p_value <= 0.01:
                stars = "**"
            elif p_value <= 0.05:
                stars = "*"
            else:
                stars = ""
                
            p_value_matrix.loc[model_a, model_b] = f"{p_value:10.3e}"
            stars_matrix.loc[model_a, model_b] = stars

    return p_value_matrix, stars_matrix

In [ ]:
def compute_win_probabilities(scores_dict: dict, nu: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    models = list(scores_dict.keys())
    models.sort(key=lambda m: scores_dict[m]["score"], reverse=True)
    
    n_models = len(models)
    
    winrate_matrix = pd.DataFrame(index=models, columns=models, dtype=float)
    expected_score_matrix = pd.DataFrame(index=models, columns=models, dtype=float)
    expected_win_rates = {}
    expected_score_rates = {}

    for i, model_a in enumerate(models):
        theta_a = scores_dict[model_a]["score"]
        sum_win_probs = 0.0
        sum_expected_scores = 0.0
        
        for j, model_b in enumerate(models):
            if i == j:
                winrate_matrix.loc[model_a, model_b] = np.nan
                expected_score_matrix.loc[model_a, model_b] = np.nan
                continue
                
            theta_b = scores_dict[model_b]["score"]
            
            # Computing the Davidson denominator (which includes the tie risk)
            denom = np.exp(theta_a) + np.exp(theta_b) + nu * np.sqrt(np.exp(theta_a) * np.exp(theta_b))
            
            # Probability of A winning against B
            p_a_wins = np.exp(theta_a) / denom
            # Probability of a tie
            p_tie = (nu * np.sqrt(np.exp(theta_a) * np.exp(theta_b))) / denom
            
            expected_score_vs_b = p_a_wins + 0.5 * p_tie  # Expected score for A against B, considering ties as half a win
            
            # A = lines, B = columns
            winrate_matrix.loc[model_a, model_b] = p_a_wins
            expected_score_matrix.loc[model_a, model_b] = expected_score_vs_b
            sum_win_probs += p_a_wins
            sum_expected_scores += expected_score_vs_b

        # Computing the Expected Win Rate (Average of win probabilities against the rest of the pool)
        expected_win_rates[model_a] = sum_win_probs / (n_models - 1)
        # Computing the Expected Score Rate (Similar to Expected Win Rate but considering ties as half a win)
        expected_score_rates[model_a] = sum_expected_scores / (n_models - 1)
        
    ewr_df = pd.DataFrame.from_dict(
        expected_win_rates, 
        orient='index', 
        columns=['Expected Global Win Rate']
    ).sort_values(by='Expected Global Win Rate', ascending=False).map(lambda x: f"{x:.1%}" if pd.notnull(x) else "-")
    
    esc_df = pd.DataFrame.from_dict(
        expected_score_rates,
        orient='index',
        columns=['Expected Global Score Rate']
    ).sort_values(by='Expected Global Score Rate', ascending=False).map(lambda x: f"{x:.1%}" if pd.notnull(x) else "-")

    winrate_matrix = winrate_matrix.map(lambda x: f"{x:.1%}" if pd.notnull(x) else "-")
        
    return winrate_matrix, ewr_df, esc_df

pairwise_probs, global_ewr, global_esc = compute_win_probabilities(btd_scores, nu)

In [ ]:
global_ewr

In [ ]:
global_esc

In [ ]:
print("Likelihood of each model per line winning against all other models per column:")
pairwise_probs

In [ ]:
visualize_btd_scores(btd_scores)
p_value_matrix, stars_matrix = compute_significance_matrix(btd_scores, n_iterations=N_ITERATIONS)

In [ ]:
p_value_matrix

In [ ]:
stars_matrix

# Agreement between student and professor scores (at the role level)

## Raw agreement

In [ ]:
from collections import defaultdict

def get_full_data(input_data_path: str = INPUT_DATA_PATH) -> list[dict[str, Any]]:
    with open(input_data_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        return data

annotations_list = []

for item in get_full_data(INPUT_DATA_PATH):
    id = item.get("reaction_id")
    student_annotations = defaultdict()

    student_annotations.update({"preferred_model": item.get("preferred_model")})
    student_annotations.update({"rating": item.get("rating")})
    student_annotations.update({"comment": item.get("comment")})
    
    labels = item.get("labels", {})
    student_annotations.update(**labels)

    model_a = item.get("model_a_name", "")
    model_b = item.get("model_b_name", "")
    
    teacher_annotations: list[dict] = item.get("teacher_annotation_data", [])
    
    for teacher_annotation in teacher_annotations:
        preferred_model = teacher_annotation.get("preferred_model", "")
        teacher_annotation.update({"preferred_model": model_a if preferred_model == 'a' else model_b if preferred_model == 'b' else None})
        del teacher_annotation["id"]
        del teacher_annotation["timestamp"]
    
    annotations_list.append({
        "id": id,
        "student_annotation": student_annotations,
        "teacher_annotation": teacher_annotations
    })

print(json.dumps(annotations_list, indent=4, ensure_ascii=False))

In [ ]:
def compute_student_teacher_agreement_per_label(label: Literal['complete', 'correct', 'relevant', 'concise', 'scaffolding', 'understandable'], filtered_out_annotators: list[str] = ["96278cf1-da97-430d-a30a-ef6fad1f6279"], input_data_path: str = INPUT_DATA_PATH) -> dict[str, tuple[int, int]]:
    
    out = defaultdict(lambda: (0, 0))
    
    for annotator in [ ann for ann in get_annotators(input_data_path) if ann not in filtered_out_annotators ]:

        observed_item_count = 0
        agreement_count = 0
        
        for item in annotations_list:
            
            teacher_annotation = next(
                (ann for ann in item.get("teacher_annotation", []) if ann.get("annotator_id") == annotator), 
                None
            )
            
            if teacher_annotation is None: # If the teacher annotation is missing for this item, we skip it
                continue
            
            student_annotation = item.get("student_annotation", {})
            
            student_preferred_model = student_annotation.get("preferred_model")
            teacher_preferred_model = teacher_annotation.get("preferred_model")
            
            if student_preferred_model == teacher_preferred_model:
                observed_item_count += 1
                
                student_annotation = item.get("student_annotation", {})
                student_label = student_annotation.get(label)
                teacher_label = teacher_annotation.get(label)
                
                
                if student_label is not None and teacher_label is not None and student_label == teacher_label:
                    agreement_count += 1
        
        out[annotator] = agreement_count, observed_item_count
    
    return out

def visualize_heatmap_global_student_teacher_agreement(filtered_out_annotators: list[str] = ["96278cf1-da97-430d-a30a-ef6fad1f6279"]) -> None:
    
    annotators = [annotator for annotator in get_annotators() if annotator not in filtered_out_annotators]
    
    data = {}
    
    for label in LABELS:
        agreement_results = compute_student_teacher_agreement_per_label(label, filtered_out_annotators)
        data[label] = {}
        
        for annotator, (agreement_count, observed_item_count) in agreement_results.items():
            data[label][annotator] = (agreement_count, observed_item_count)
    
    plt.figure(figsize=(12, 6))
    heatmap_data = np.zeros((len(LABELS), len(annotators)))
    for i, label in enumerate(LABELS):
        for j, annotator in enumerate(annotators):
            if annotator in data[label]:
                agreement_count, observed_item_count = data[label][annotator]
                heatmap_data[i, j] = agreement_count / observed_item_count * 100 if observed_item_count > 0 else 0.0

    sns.heatmap(heatmap_data, annot=True, fmt='.1f', xticklabels=False, yticklabels=LABELS, cmap=sns.color_palette("ch:s=-.5,r=.1,d=.2,l=.8", as_cmap=True), cbar_kws={'label': 'Agreement (%)'})
    plt.title("Student-Teacher Agreement")
    plt.xlabel("Teachers")
    plt.ylabel("Labels")
    plt.tight_layout()
    plt.show()

def compute_global_student_teacher_agreement(filtered_out_annotators: list[str] = ["96278cf1-da97-430d-a30a-ef6fad1f6279"], input_data_path: str = INPUT_DATA_PATH) -> dict[str, dict[str, float]]:
    out = {}
    
    for label in LABELS:
        agreement_results = compute_student_teacher_agreement_per_label(label, filtered_out_annotators, input_data_path)
        print(f" Agreements for label '{label}': ".center(60, '='))
        for annotator, (agreement_count, observed_item_count) in agreement_results.items():
            percentage = agreement_count / observed_item_count * 100 if observed_item_count > 0 else 0.0
            print(f"{annotator}: {agreement_count}/{observed_item_count} ({percentage:.2f}%)".center(60))
            
            if annotator not in out:
                out[annotator] = [0, 0]
            out[annotator][0] += agreement_count
            out[annotator][1] += observed_item_count
        print('\n')
    
    print('='*60, '\n', " Global Student-Teacher Agreement: ".center(60, '='), '\n', '='*60)
    
    for annotator, (total_agreement, total_observed) in out.items():
        percentage = total_agreement / total_observed * 100 if total_observed > 0 else 0.0
        print(f"{annotator}: {total_agreement}/{total_observed} ({percentage:.2f}%)".center(60))
    
    return out

global_agreement_values = compute_global_student_teacher_agreement()
visualize_heatmap_global_student_teacher_agreement()

In [ ]:
def compute_student_teacher_agreement_per_metric(metric: Literal['preferred_model', 'rating'] = "preferred_model", filtered_out_annotators: list[str] = ["96278cf1-da97-430d-a30a-ef6fad1f6279"], input_data_path: str = INPUT_DATA_PATH) -> dict[str, tuple[int, int]]:
    
    if metric not in ['rating', 'preferred_model']:
        raise ValueError(f"Invalid metric '{metric}'. Expected 'rating' or 'preferred_model'.")
    
    out = defaultdict(lambda: (0, 0))
    
    for annotator in [ ann for ann in get_annotators(input_data_path) if ann not in filtered_out_annotators ]:

        observed_item_count = 0
        agreement_count = 0
        
        for item in annotations_list:
            
            teacher_annotation = next(
                (ann for ann in item.get("teacher_annotation", []) if ann.get("annotator_id") == annotator), 
                None
            )
            
            if teacher_annotation is None: # If the teacher annotation is missing for this item, we skip it
                continue
                        
            student_annotation = item.get("student_annotation", {})
            
            student_preferred_model = student_annotation.get("preferred_model")
            teacher_preferred_model = teacher_annotation.get("preferred_model")
                
            if metric == "preferred_model":
                student_metric = student_preferred_model
                teacher_metric = teacher_preferred_model
                
                observed_item_count += 1
                
                if student_metric == teacher_metric:
                    agreement_count += 1
            
            else :# metric == "rating":
                student_metric = student_annotation.get("rating")
                teacher_metric = teacher_annotation.get("rating")
                
                if student_preferred_model == teacher_preferred_model:
                    observed_item_count += 1
                    if student_metric and teacher_metric and student_metric == teacher_metric:
                        agreement_count += 1
        
        out[annotator] = agreement_count, observed_item_count
        
    print('='*60, '\n', f" Student-Teacher Agreement ({metric}): ".center(60, '='), '\n', '='*60)
    
    for annotator, (total_agreement, total_observed) in out.items():
        percentage = total_agreement / total_observed * 100 if total_observed > 0 else 0.0
        print(f"{annotator}: {total_agreement}/{total_observed} ({percentage:.2f}%)".center(60))
    
    return out

In [ ]:
preferred_model_agreement = compute_student_teacher_agreement_per_metric("preferred_model")
rating_agreement = compute_student_teacher_agreement_per_metric("rating")

## Chance-corrected agreement (Gwet's AC1)

In [ ]:
from irrCAC.raw import CAC

def get_agreement_and_ac1_data(filtered_out_annotators: list[str] = ["96278cf1-da97-430d-a30a-ef6fad1f6279"]):
    annotators = [ann for ann in get_annotators() if ann not in filtered_out_annotators]
    
    data_matrix = {}
    
    for label in LABELS:
        data_matrix[label] = {}
        
        for annotator in annotators:
            student_votes = []
            teacher_votes = []
            
            for item in annotations_list:
                teacher_ann = next(
                    (ann for ann in item.get("teacher_annotation", []) if ann.get("annotator_id") == annotator),
                    None
                )
                
                if teacher_ann is None:
                    continue
                
                student_ann = item.get("student_annotation", {})
                
                student_pref = student_ann.get("preferred_model")
                teacher_pref = teacher_ann.get("preferred_model")
                
                if student_pref != teacher_pref:
                    continue # We only consider items where the student and teacher agreed on the preferred model for this item
                    
                student_label = student_ann.get(label)
                teacher_label = teacher_ann.get(label)
                
                if student_label is not None and teacher_label is not None:
                    # Explicit conversion to string to ensure compatibility with irrCAC
                    student_votes.append(str(student_label))
                    teacher_votes.append(str(teacher_label))
            
            n = len(student_votes)
            
            if n > 1: # At least two votes are needed to compute agreement and AC1
                # Raw agreement percentage
                s_array = np.array(student_votes)
                t_array = np.array(teacher_votes)
                raw_agreement = np.sum(s_array == t_array) / n * 100
                
                # AC1 score
                df_ratings = pd.DataFrame({
                    'Student': student_votes,
                    'Teacher': teacher_votes
                })
                
                cac = CAC(df_ratings)

                try:
                    ac1_results = cac.gwet()
                    
                    ac1_score = ac1_results['est']['coefficient_value']
                    ci = ac1_results['est']['confidence_interval']
                    p_value = ac1_results['est']['p_value']
                except Exception as e:
                    print(e)
                    ac1_score = 1.0 if raw_agreement == 100.0 else 0.0
                    ci = (np.nan, np.nan)
                    p_value = np.nan
                
            else:
                raw_agreement = 0.0
                ac1_score = np.nan
                ci = (np.nan, np.nan)
                p_value = np.nan
                
            data_matrix[label][annotator] = {
                "raw_percent": raw_agreement,
                "ac1": ac1_score,
                "ci_95": ci,
                "p_value": p_value,
            }
            
    return data_matrix, annotators

def visualize_heatmap_ac1_and_agreement(data_matrix: dict, annotators: list[str], labels: list[str] = LABELS) -> None:
    """
    Show a heatmap of the AC1 scores, significance, and raw agreement percentages for each label and annotator.
    """
    # Initialising matrices (colours for AC1, text pour display)
    ac1_data = np.zeros((len(labels), len(annotators)))
    annot_data = np.empty((len(labels), len(annotators)), dtype=object)

    for i, label in enumerate(labels):
        for j, annotator in enumerate(annotators):
            stats = data_matrix[label].get(annotator, {})
            
            ac1 = stats.get("ac1", np.nan)
            raw_pct = stats.get("raw_percent", 0.0)
            p_val = stats.get("p_value", 1.0)
            
            # Colour matrix (we set to 0 if score calculation failed)
            ac1_data[i, j] = ac1 if not np.isnan(ac1) else 0.0
            
            # Significance stars
            stars = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
            
            # Formatting text within the heatmap cell: AC1 score with 2 decimals, significance stars, and raw agreement percentage with 1 decimal
            if not np.isnan(ac1):
                annot_data[i, j] = f"{ac1:.3f} {'(' + stars + ')' if stars else ''}\n({raw_pct:.1f}%)"
            else:
                annot_data[i, j] = f"N/A\n({raw_pct:.1f}%)"

    # Cleaning of identifiers (to avoid long UUIDs from distorting the X-axis)
    clean_annotators = [f"Prof {k+1}\n({ann[:8]})" for k, ann in enumerate(annotators)]

    plt.figure(figsize=(10, 6))
    
    ax = sns.heatmap(
        ac1_data,
        annot=annot_data,
        fmt="",
        xticklabels=clean_annotators,
        yticklabels=labels,
        cmap=sns.color_palette("ch:s=-.5,r=.1,d=.2,l=.8", as_cmap=True),
        vmin=0.0, vmax=1.0,
        cbar_kws={'label': "Agreement Score (Gwet's AC1)"}
    )
    
    for text in ax.texts:
        text.set_fontsize(11)
        if "***" in text.get_text():
            text.set_fontweight(1000)
        elif "**" in text.get_text():
            text.set_fontweight(800)
        elif "*" in text.get_text():
            text.set_fontweight(600)
        else:
            text.set_fontsize(10)
    
    plt.title("Student-Teacher Agreement (AC1 & Raw Agreement)")
    plt.xlabel("Teachers")
    plt.ylabel("Evaluation Criteria")
    
    plt.tight_layout()
    plt.show()

In [ ]:
matrix, annotators = get_agreement_and_ac1_data()
for annotator in annotators:
    print(f"Annotator: {annotator}")
    for label in LABELS:
        agreement_data = matrix[label][annotator]
        print(f"Label: {label}".center(25) + f" | Raw Agreement: {agreement_data['raw_percent']:.2f}%".center(15) + f" | AC1: {agreement_data['ac1']:.3f}".center(10) + f" | 95% CI: ({agreement_data['ci_95'][0]:.3f}, {agreement_data['ci_95'][1]:.3f})".center(15) + f" | p: {agreement_data['p_value']:.2e}".center(20))
    print("\n")

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'Tahoma', 'DejaVu Sans']
visualize_heatmap_ac1_and_agreement(matrix, annotators)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from irrCAC.raw import CAC

def get_rating_agreement_and_ac2_data(
    filtered_out_annotators: list[str] = ["96278cf1-da97-430d-a30a-ef6fad1f6279"],
    weight_type: str = "ordinal"
):
    annotators = [ann for ann in get_annotators() if ann not in filtered_out_annotators]
    results = {}
    
    for annotator in annotators:
        student_votes = []
        teacher_votes = []
        
        for item in annotations_list:
            teacher_list = item.get("teacher_annotation", [])
            teacher_ann = next((ann for ann in teacher_list if ann.get("annotator_id") == annotator), None)
            
            if teacher_ann is None:
                continue
            
            student_ann = item.get("student_annotation", {})
            
            s_pref = student_ann.get("preferred_model")
            t_pref = teacher_ann.get("preferred_model")
            
            # Condition stricte : consensus réel sur le modèle gagnant (exclut None)
            if s_pref is None or t_pref is None or s_pref != t_pref:
                continue
                
            s_val = student_ann.get("rating")
            t_val = teacher_ann.get("rating")
            
            if s_val is not None and t_val is not None:
                try:
                    student_votes.append(int(round(float(s_val))))
                    teacher_votes.append(int(round(float(t_val))))
                except (ValueError, TypeError):
                    continue
        
        n = len(student_votes)
        
        if n > 1:
            s_array = np.array(student_votes)
            t_array = np.array(teacher_votes)
            
            # Exact agreement (same rating)
            exact_agreement = np.sum(s_array == t_array) / n * 100
            
            # Adjacent agreement (tolerance of +/- 1 point)
            adjacent_agreement = np.sum(np.abs(s_array - t_array) <= 1) / n * 100
            
            # Gwet's AC2
            df_ratings = pd.DataFrame({
                'Student': student_votes,
                'Teacher': teacher_votes
            })
            
            cac = CAC(df_ratings, weights=weight_type, categories=[1, 2, 3, 4, 5])

            try:
                ac2_results = cac.gwet()
                ac2_score = ac2_results['est']['coefficient_value']
                p_val = ac2_results['est']['p_value']
                ci = ac2_results['est']['confidence_interval']
            except Exception as e:
                print(f"Error computing AC2 for {annotator[:8]} : {e}")
                ac2_score = 1.0 if exact_agreement == 100.0 else 0.0
                p_val = np.nan
                ci = [np.nan, np.nan]
        else:
            exact_agreement, adjacent_agreement = 0.0, 0.0
            ac2_score, p_val = np.nan, np.nan
            ci = [np.nan, np.nan]
            
        results[annotator] = {
            "n_items": n,
            "exact_percent": exact_agreement,
            "adjacent_percent": adjacent_agreement,
            "ac2": ac2_score,
            "p_value": p_val,
            "ci_95": ci,
            "student_scores": student_votes,
            "teacher_scores": teacher_votes
        }
        
    return results, annotators


def visualize_ratings_ac2_heatmap(results: dict, annotators: list[str]) -> None:
    ac2_data = np.zeros((1, len(annotators)))
    annot_data = np.empty((1, len(annotators)), dtype=object)

    for j, annotator in enumerate(annotators):
        stats = results.get(annotator, {})
        ac2 = stats.get("ac2", np.nan)
        exact_pct = stats.get("exact_percent", 0.0)
        adj_pct = stats.get("adjacent_percent", 0.0)
        p_val = stats.get("p_value", 1.0)
        n = stats.get("n_items", 0)
        
        ac2_data[0, j] = ac2 if not np.isnan(ac2) else 0.0
        
        stars = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
        
        if not np.isnan(ac2):
            annot_data[0, j] = (
                f"AC2: {ac2:.3f} {stars}\n"
                f"Exact: {exact_pct:.1f}%\n"
                f"Adj (+/-1): {adj_pct:.1f}%\n"
                f"(N = {n})"
            )
        else:
            annot_data[0, j] = f"N/A\n(N = {n})"

    clean_annotators = [f"Prof {k+1}\n({ann[:8]})" for k, ann in enumerate(annotators)]

    plt.figure(figsize=(max(7, len(annotators) * 3), 3.5))
    
    ax = sns.heatmap(
        ac2_data,
        annot=annot_data,
        fmt="",
        xticklabels=clean_annotators,
        yticklabels=["Likert Rating\n(1-5 Scale)"],
        cmap=sns.color_palette("ch:s=-.5,r=.1,d=.2,l=.8", as_cmap=True),
        vmin=0.0, vmax=1.0,
        cbar_kws={'label': "Gwet's AC2 (Ordinal)"}
    )
    
    for text in ax.texts:
        text.set_fontsize(10)
        if any(s in text.get_text() for s in ["***", "**", "*"]):
            text.set_fontweight("bold")
            
    plt.title("Student-Teacher Agreement on Pointwise Ratings (Consensus Turns)", pad=15)
    plt.xlabel("Teachers")
    plt.tight_layout()
    plt.show()


# --- Exécution ---
ratings_matrix, annotators = get_rating_agreement_and_ac2_data()

print("\n=== Results for AC2 on pointwise ratings ===")
for annotator in annotators:
    res = ratings_matrix[annotator]
    ci_str = f"[{res['ci_95'][0]:.3f}, {res['ci_95'][1]:.3f}]" if not np.isnan(res['ci_95'][0]) else "N/A"
    print(
        f"Annotator: {annotator[:8]}... | N = {res['n_items']} | "
        f"Exact: {res['exact_percent']:.1f}% | Adj (+/-1): {res['adjacent_percent']:.1f}% | "
        f"AC2: {res['ac2']:.3f} | 95% CI: {ci_str} | p: {res['p_value']:.2e}"
    )

visualize_ratings_ac2_heatmap(ratings_matrix, annotators)

In [ ]:
def get_llm_as_a_judge_annotations_list(jsonl_path: str) -> list[tuple[str, str, str]]:
    """
    - Parse the LLM-as-a-judge annotation JSON data and return it in a format suitable for bootstrapping analysis with BTD.
    - Ditch cases where at least one of the two positions is missing (_i.e._, only one of the two models was rated by the LLM-as-a-judge framework) -- preventing from taking into account position bias in the analysis.
    
    > [NOTE] _Addressing position swapping_ (An et al., 2025):
        - In the original LLM-as-a-judge paper (Zheng et al., 2023), it is advised to resolve inconsistencies in model ratings (_i.e._ cases when the LLM-as-a-judge framework rates the models inconsistently based on their position) with a tie.
        - However, because our rating model already allows for ties, this would inevitably skew the BTD parametres (nu in particular).
        - Hence, we integrate each of the swapped annotations into the final list. In that configuration, a win and a loss will be treated as roughly equivalent to a tie, giving more weight to annotations that are consistent across positions.
        
    Returns:
        A list of tuples (model_a, model_b, choice) where choice is 'a', 'b', or 'both_equal'
    """
    with open(jsonl_path, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f]
        
    grouped = defaultdict(list)
    for r in records:
        key = (r.get("reaction_id"), r.get("annotator_id"))
        grouped[key].append(r)
    
    out = []
    
    for key, group in grouped.items():
        
        if len(group) != 2: # One of the two positions is missing, we skip this record
            print(f"Skipping record {key} due to missing a position.")
            continue
        
        for r in group:
            model_a = r.get("model_a_name", "")
            model_b = r.get("model_b_name", "")
            
            pref = r.get("judge_preferred_model", "")
            
            if not model_a or not model_b or not pref:
                print(f"Missing required fields in record: {r}")
                continue
            
            if pref == model_a:
                choice = 'a'
            elif pref == model_b:
                choice = 'b'
            elif pref == "both_equal":
                choice = 'both_equal'
            else:
                print(f"Invalid judge_preferred_model '{pref}' for models '{model_a}' and '{model_b}'. Expected one of: '{model_a}', '{model_b}', or 'both_equal'.")
                continue

            out.append((model_a, model_b, choice))

    print(f"Extracted {len(out)} annotations ({len(out) // 2} pairs) ready for analysis.")
    return out

def run_btd_pipeline_llm_as_a_judge(input_data_path: str, n_iterations: int = 1000, seed: int = 2026) -> tuple[list[float], float]:
    llm_annotations = get_llm_as_a_judge_annotations_list(input_data_path)

    llm_btd_scores, llm_nu = compute_btd_with_ci(llm_annotations, n_iterations=n_iterations, seed=seed)

    print(f"Estimated Davidson tie probability parameter (nu): {llm_nu:.3f}")
    visualize_btd_scores(llm_btd_scores)

    llm_pairwise_probs, llm_global_ewr, llm_global_esc = compute_win_probabilities(llm_btd_scores, llm_nu)
    llm_p_value_matrix, llm_stars_matrix = compute_significance_matrix(llm_btd_scores)

    print(" Pairwise Win Probabilities: ".center(60, '='))
    display(llm_pairwise_probs)
    
    print(" Global Expected Win Rates: ".center(60, '='))
    display(llm_global_ewr)

    print(" Global Expected Score Rates: ".center(60, '='))
    display(llm_global_esc)

    print(" p-Value Matrix: ".center(60, '='))
    display(llm_p_value_matrix)
    
    print(" Significance Stars Matrix: ".center(60, '='))
    display(llm_stars_matrix)
    
    return llm_btd_scores, llm_nu

### Phi4 on the student conversation dataset

In [ ]:
llm_phi4_reactions_scores, llm_phi4_reactions_nu = run_btd_pipeline_llm_as_a_judge(os.path.join("..", "data", "judge_runs", "reactions__phi4.jsonl"), n_iterations=1000)

### Qwen3-235B-A22B on the student conversation dataset

In [ ]:
llm_qwen3_235b_reactions_scores, llm_qwen3_235b_reactions_nu = run_btd_pipeline_llm_as_a_judge(os.path.join("..", "data", "judge_runs", "reactions__qwen3-235b.jsonl"), n_iterations=1000)

### Phi4 on the synthetic dataset

In [ ]:
llm_profs_phase2_phi4_scores, llm_profs_phase2_phi4_nu = run_btd_pipeline_llm_as_a_judge(os.path.join("..", "data", "judge_runs", "profs_phase2__phi4.jsonl"), n_iterations=1000)

### Qwen3-235B-A22B on the synthetic dataset

In [ ]:
llm_profs_phase2_qwen3_235b_scores, llm_profs_phase2_qwen3_235b_nu = run_btd_pipeline_llm_as_a_judge(os.path.join("..", "data", "judge_runs", "profs_phase2__qwen3-235b.jsonl"), n_iterations=1000)

### Rank correlation between human- and LLM-estimated BTD logit scores

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

def compute_probabilistic_correlations(human_btd_scores: dict, llm_btd_scores: dict, model_name: str, n_iterations: int = 10000, seed: int = 2026) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute Spearman and Kendall correlations while accounting for the uncertainty of the BTD scores via a Monte Carlo simulation on the bootstrap arrays.
    """
    models = list(human_btd_scores.keys())
    
    spearman_corrs = []
    kendall_corrs = []
    
    n_human_samples = len(human_btd_scores[models[0]]["bootstrapped_scores_array"])
    n_llm_samples = len(llm_btd_scores[models[0]]["bootstrapped_scores_array"])
    
    # Ensure we don't exceed the number of available bootstrap samples
    max_samples = min(n_human_samples, n_llm_samples)
    if n_iterations > max_samples:
        print(f"Warning: n_iterations ({n_iterations}) was reduced to the number of available bootstrap samples ({max_samples}).")
        n_iterations = max_samples

    rng = np.random.default_rng(seed)
    
    # If bootstrapped arrays are of different lengths, we sample indices from the smaller one to ensure paired comparisons
    # We sample indices with replacement to allow for more iterations than available samples
    sampled_indices = rng.choice(max_samples, size=n_iterations, replace=True)

    for idx in tqdm(sampled_indices, desc="Monte-Carlo Correlations"):
        # Reconstruct scores from raw logits for this iteration using the PAIRED index
        h_scores = [human_btd_scores[m]["bootstrapped_scores_array"][idx] for m in models]
        l_scores = [llm_btd_scores[m]["bootstrapped_scores_array"][idx] for m in models]
        
        # Ignore iterations containing NaN in scores
        if np.isnan(h_scores).any() or np.isnan(l_scores).any():
            continue
            
        # Compute Spearman and Kendall correlations for this specific draw
        with np.errstate(all='ignore'):
            s_corr, _ = stats.spearmanr(h_scores, l_scores)
            k_corr, _ = stats.kendalltau(h_scores, l_scores)
        
        # Ignore iterations if the correlation returns a NaN (e.g., constant scores)
        if np.isnan(s_corr) or np.isnan(k_corr):
            continue
            
        spearman_corrs.append(s_corr)
        kendall_corrs.append(k_corr)
        
    spearman_corrs = np.array(spearman_corrs)
    kendall_corrs = np.array(kendall_corrs)
    
    stats_spearman = {
        "median": np.nanmedian(spearman_corrs),
        "ci_lower": np.nanpercentile(spearman_corrs, 2.5),
        "ci_upper": np.nanpercentile(spearman_corrs, 97.5)
    }
    stats_kendall = {
        "median": np.nanmedian(kendall_corrs),
        "ci_lower": np.nanpercentile(kendall_corrs, 2.5),
        "ci_upper": np.nanpercentile(kendall_corrs, 97.5)
    }
        
    print("="*50)
    print(f"Spearman's Rho: {stats_spearman['median']:.3f} CI 95% [{stats_spearman['ci_lower']:.3f}, {stats_spearman['ci_upper']:.3f}]")
    print(f"Kendall's Tau: {stats_kendall['median']:.3f} CI 95% [{stats_kendall['ci_lower']:.3f}, {stats_kendall['ci_upper']:.3f}]")
    print("="*50)
    
    # Dataviz
    plt.figure(figsize=(10, 6))
    sns.kdeplot(spearman_corrs, fill=True, color="#1f77b4", alpha=0.4, label=f"Spearman's Rho (Median: {stats_spearman['median']:.2f})")
    plt.axvline(stats_spearman['median'], color="#1f77b4", linestyle="-")
    
    sns.kdeplot(kendall_corrs, fill=True, color="#ff7f0e", alpha=0.4, label=f"Kendall's Tau (Median: {stats_kendall['median']:.2f})")
    plt.axvline(stats_kendall['median'], color="#ff7f0e", linestyle="-")
    
    plt.axvline(0, color="gray", linestyle=":", linewidth=2, label="Absolute independence")
    plt.title(f"Probabilistic Distribution of Human vs LLM Alignment ({model_name})")
    plt.xlabel("Correlation Coefficient")
    plt.ylabel("Probability Density (Monte-Carlo)")
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    return spearman_corrs, kendall_corrs

In [ ]:
spearman_dist, kendall_dist = compute_probabilistic_correlations(btd_scores, llm_phi4_reactions_scores, model_name="Phi4", n_iterations=10000)

In [ ]:
spearman_dist, kendall_dist = compute_probabilistic_correlations(btd_scores, llm_qwen3_235b_reactions_scores, model_name="Qwen3-235B-A22B", n_iterations=10000)

# Analysis of ratings

In [ ]:
from pathlib import Path

def _load_json_or_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        if path.suffix == ".jsonl":
            return [json.loads(line) for line in f if line.strip()]
        return json.load(f)

def _resolve_model_name(preferred_model, model_a_name: str, model_b_name: str) -> str | None:
    preferred_model_str = str(preferred_model).strip()
    if preferred_model_str in {"a", model_a_name}:
        return model_a_name
    if preferred_model_str in {"b", model_b_name}:
        return model_b_name
    return None

def _extract_rating_rows(dataset_path: Path) -> list[dict[str, object]]:
    payload = _load_json_or_jsonl(dataset_path)
    if not isinstance(payload, list):
        return []

    has_teacher_annotations = any(
        isinstance(item, dict) and bool(item.get("teacher_annotation_data"))
        for item in payload
    )
    if not has_teacher_annotations:
        return []

    rows = []
    dataset_name = dataset_path.stem

    for item in payload:
        if not isinstance(item, dict):
            continue

        model_a_name = item.get("model_a_name")
        model_b_name = item.get("model_b_name")
        if not model_a_name or not model_b_name:
            continue

        student_rating = item.get("rating")
        student_model = _resolve_model_name(item.get("preferred_model"), model_a_name, model_b_name)
        if student_model is not None and student_rating not in (None, 0):
            rows.append({
                "dataset": dataset_name,
                "role": "student",
                "model": student_model,
                "rating": float(student_rating),
            })

        for teacher_annotation in item.get("teacher_annotation_data", []):
            if not isinstance(teacher_annotation, dict):
                continue

            teacher_rating = teacher_annotation.get("rating")
            teacher_model = _resolve_model_name(teacher_annotation.get("preferred_model"), model_a_name, model_b_name)
            if teacher_model is not None and teacher_rating not in (None, 0):
                rows.append({
                    "dataset": dataset_name,
                    "role": "teacher",
                    "model": teacher_model,
                    "rating": float(teacher_rating),
                })

    return rows

def compute_average_rating_per_dataset(data_dir: str = "../data") -> pd.DataFrame:
    data_dir_path = Path(data_dir)
    dataset_paths = sorted(
        path
        for path in data_dir_path.rglob("*")
        if path.is_file() and path.suffix in {".json", ".jsonl"} and "judge_runs" not in path.parts
    )

    rating_rows = []
    for path in dataset_paths:
        try:
            rating_rows.extend(_extract_rating_rows(path))
        except Exception as exc:
            print(f"Skipping {path.name}: {exc}")

    if not rating_rows:
        raise ValueError("No student or teacher ratings were found in the student conversation datasets.")

    ratings_df = pd.DataFrame(rating_rows)
    summary_df = (
        ratings_df
        .groupby(["dataset", "role", "model"], as_index=False)
        .agg(
            average_rating=("rating", "mean"),
            n_ratings=("rating", "size"),
        )
        .sort_values(["dataset", "role", "average_rating"], ascending=[True, True, False])
        .reset_index(drop=True)
    )

    print("Average rating per model, split by dataset and role")
    print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

    return summary_df

average_ratings_by_dataset = compute_average_rating_per_dataset()
average_ratings_by_dataset

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats

def compute_btd_win_rates(scores_dict: dict, nu: float) -> pd.DataFrame:
    models = list(scores_dict.keys())
    models.sort(key=lambda model: scores_dict[model]["score"], reverse=True)

    rows = []
    for model_a in models:
        theta_a = scores_dict[model_a]["score"]
        sum_win_probs = 0.0

        for model_b in models:
            if model_a == model_b:
                continue

            theta_b = scores_dict[model_b]["score"]
            denom = np.exp(theta_a) + np.exp(theta_b) + nu * np.sqrt(np.exp(theta_a) * np.exp(theta_b))
            win_probability = np.exp(theta_a) / denom
            sum_win_probs += win_probability

        expected_win_rate = sum_win_probs / (len(models) - 1) if len(models) > 1 else np.nan
        rows.append({
            "model": model_a,
            "btd_win_rate": expected_win_rate,
        })

    return pd.DataFrame(rows)

def compare_btd_to_average_ratings(average_df: pd.DataFrame, scores_dict: dict, nu: float, dataset_name: str) -> pd.DataFrame:
    dataset_df = average_df[average_df["dataset"] == dataset_name].copy()
    btd_df = compute_btd_win_rates(scores_dict, nu)

    merged = dataset_df.merge(btd_df, on="model", how="inner")
    if merged.empty:
        raise ValueError(f"No overlapping models were found for dataset '{dataset_name}'.")

    print(f"BTD vs average ratings for dataset: {dataset_name}")
    print(merged.sort_values(["role", "model"]).to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    print()

    test_rows = []
    for role, role_df in merged.groupby("role"):
        role_df = role_df.sort_values("model")
        rating_values = role_df["average_rating"].to_numpy()
        btd_values = role_df["btd_win_rate"].to_numpy()

        if len(role_df) < 3:
            test_name = "not enough pairs for spearman"
            statistic = np.nan
            p_value = np.nan
        else:
            try:
                test_result = stats.spearmanr(rating_values, btd_values, nan_policy="omit")
                test_name = "Spearman rank correlation"
                statistic = float(test_result.statistic)
                p_value = float(test_result.pvalue)
            except ValueError:
                test_name = "spearman failed"
                statistic = np.nan
                p_value = np.nan

        test_rows.append({
            "dataset": dataset_name,
            "role": role,
            "test": test_name,
            "spearman_rho": statistic,
            "p_value": p_value,
            "n_models": len(role_df),
            "significant_at_0.05": bool(p_value < 0.05) if not np.isnan(p_value) else False,
        })

    results_df = pd.DataFrame(test_rows).sort_values(["dataset", "role"]).reset_index(drop=True)
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    return results_df

current_dataset_name = Path(INPUT_DATA_PATH).stem
btd_comparison_results = compare_btd_to_average_ratings(average_ratings_by_dataset, btd_scores, nu, current_dataset_name)
btd_comparison_results

In [ ]:
from pathlib import Path
import pandas as pd
import scipy.stats as stats
import numpy as np

def evaluate_judge_run_vs_ratings(
    judge_jsonl_path: str, 
    average_df: pd.DataFrame, 
    dataset_name: str, 
    n_iterations: int = 1000
) -> pd.DataFrame:
    """
    Runs the BTD pipeline on an LLM judge run file (.jsonl),
    then compares the obtained ranks with the average human ratings (students and teachers)
    via Spearman correlation.
    """
    llm_annotations = get_llm_as_a_judge_annotations_list(judge_jsonl_path)
    
    btd_scores_dict, nu_param = compute_btd_with_ci(llm_annotations, n_iterations=n_iterations)
    
    models = list(btd_scores_dict.keys())
    models.sort(key=lambda m: btd_scores_dict[m]["score"], reverse=True)
    
    btd_rows = []
    for model_a in models:
        theta_a = btd_scores_dict[model_a]["score"]
        sum_win_probs = 0.0
        for model_b in models:
            if model_a == model_b:
                continue
            theta_b = btd_scores_dict[model_b]["score"]
            denom = np.exp(theta_a) + np.exp(theta_b) + nu_param * np.sqrt(np.exp(theta_a) * np.exp(theta_b))
            win_probability = np.exp(theta_a) / denom
            sum_win_probs += win_probability
        
        expected_win_rate = sum_win_probs / (len(models) - 1) if len(models) > 1 else np.nan
        btd_rows.append({"model": model_a, "btd_win_rate": expected_win_rate})
    
    btd_df = pd.DataFrame(btd_rows)
    
    # Merging the BTD results with the average ratings for the specified dataset
    dataset_df = average_df[average_df["dataset"] == dataset_name].copy()
    merged = dataset_df.merge(btd_df, on="model", how="inner")
    
    if merged.empty:
        raise ValueError(f"No overlapping models were found between the dataset '{dataset_name}' and the run {Path(judge_jsonl_path).name}.")

    print(f"=== Judge Run: {Path(judge_jsonl_path).name} vs Human Ratings ({dataset_name}) ===")
    
    # Role-wise correlation analysis
    test_rows = []
    for role, role_df in merged.groupby("role"):
        role_df = role_df.sort_values("model")
        rating_values = role_df["average_rating"].to_numpy()
        btd_values = role_df["btd_win_rate"].to_numpy()

        if len(role_df) < 3:
            statistic, p_value = np.nan, np.nan
        else:
            try:
                test_result = stats.spearmanr(rating_values, btd_values, nan_policy="omit")
                statistic, p_value = float(test_result.statistic), float(test_result.pvalue)
            except ValueError:
                statistic, p_value = np.nan, np.nan

        test_rows.append({
            "judge_run": Path(judge_jsonl_path).name,
            "dataset": dataset_name,
            "role": role,
            "spearman_rho": statistic,
            "p_value": p_value,
            "n_models": len(role_df),
            "significant_at_0.05": bool(p_value < 0.05) if not np.isnan(p_value) else False,
        })

    results_df = pd.DataFrame(test_rows)
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("\n" + "="*70 + "\n")
    return results_df

target_dataset = "students_teacher_gold" # ou ton autre nom de dataset

phi4_results = evaluate_judge_run_vs_ratings(
    os.path.join("..", "data", "judge_runs", "profs_phase2__phi4.jsonl"), 
    average_ratings_by_dataset, 
    target_dataset
)

qwen_results = evaluate_judge_run_vs_ratings(
    os.path.join("..", "data", "judge_runs", "profs_phase2__qwen3-235b.jsonl"), 
    average_ratings_by_dataset, 
    target_dataset
)

## Comparing ratings (learners vs. teachers)

## RQ1

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = "../data/students_teacher_gold.json"

from scipy.stats import binomtest

def test_position_bias(l_counts, t_counts):
    print("\n" + "="*60)
    print("POSITION BIAS TEST (Model A vs Model B)")
    print("="*60)
    
    # Learners
    l_a = l_counts["a"]
    l_b = l_counts["b"]
    l_total_decisive = l_a + l_b
    
    if l_total_decisive > 0:
        p_l = binomtest(l_a, l_total_decisive, 0.5, alternative='two-sided').pvalue
        sig_l = "***" if p_l < 0.001 else "**" if p_l < 0.01 else "*" if p_l < 0.05 else "ns"
        print(f"LEARNERS: A={l_a}, B={l_b} (Total decisive: {l_total_decisive})")
        print(f"Probability of choosing A: {(l_a/l_total_decisive)*100:.1f}%")
        print(f"Binomial test p-value: {p_l:.5f} {sig_l}")
    
    print("-" * 60)
    
    # Teachers
    t_a = t_counts["a"]
    t_b = t_counts["b"]
    t_total_decisive = t_a + t_b
    
    if t_total_decisive > 0:
        p_t = binomtest(t_a, t_total_decisive, 0.5, alternative='two-sided').pvalue
        sig_t = "***" if p_t < 0.001 else "**" if p_t < 0.01 else "*" if p_t < 0.05 else "ns"
        print(f"TEACHERS: A={t_a}, B={t_b} (Total decisive: {t_total_decisive})")
        print(f"Probability of choosing A: {(t_a/t_total_decisive)*100:.1f}%")
        print(f"Binomial test p-value: {p_t:.5f} {sig_t}")
    print("="*60)

# After loading data
# test_position_bias(l_counts, t_counts)

def load_preference_data():
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    l_counts = {"a": 0, "both_equal": 0, "b": 0}
    t_counts = {"a": 0, "both_equal": 0, "b": 0}
    
    l_total = 0
    t_total = 0
    
    for item in data:
        l_pref = item.get("preferred_model")
        model_a = item.get("model_a_name")
        model_b = item.get("model_b_name")
        
        if l_pref:
            l_total += 1
            # We check if the preference matches model_a or model_b, or if it's a tie
            if l_pref == model_a or l_pref in ["a", "model_a"]:
                l_counts["a"] += 1
            elif l_pref == model_b or l_pref in ["b", "model_b"]:
                l_counts["b"] += 1
            else:
                l_counts["both_equal"] += 1
                
        for ann in item.get("teacher_annotation_data", []):
            t_pref = ann.get("preferred_model")
            if t_pref:
                t_total += 1
                p_lower = str(t_pref).lower()
                if p_lower in ["a", "model_a", "model a"]:
                    t_counts["a"] += 1
                elif p_lower in ["b", "model_b", "model b"]:
                    t_counts["b"] += 1
                else:
                    t_counts["both_equal"] += 1
                    
    return l_counts, t_counts, l_total, t_total

from scipy.stats import binomtest, fisher_exact

def test_tie_propensity(l_counts, t_counts):
    print("\n" + "="*60)
    print("TIE PROPENSITY TEST (Fisher's Exact Test)")
    print("="*60)
    
    l_decisive = l_counts["a"] + l_counts["b"]
    l_ties = l_counts["both_equal"]
    
    t_decisive = t_counts["a"] + t_counts["b"]
    t_ties = t_counts["both_equal"]
    
    table = [
        [l_decisive, l_ties],
        [t_decisive, t_ties]
    ]
    
    oddsratio, p_value = fisher_exact(table)
    
    sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
    
    print(f"Learners - Decisive: {l_decisive:<3} | Ties: {l_ties}")
    print(f"Teachers - Decisive: {t_decisive:<3} | Ties: {t_ties}")
    print(f"Fisher's exact test p-value: {p_value:.5e} {sig} -- Value: {oddsratio:.3f}")
    print("="*60)

def plot_preferences_distribution(l_counts, t_counts, l_total, t_total):
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(10, 6))

    categories = ["a", "both_equal", "b"]
    plot_labels = ["Model A Preferred", "Tie (Both Equal)", "Model B Preferred"]
    
    x = np.arange(len(categories))
    width = 0.35

    l_pct = [(l_counts[c] / l_total) * 100 for c in categories]
    t_pct = [(t_counts[c] / t_total) * 100 for c in categories]

    l_errors = [1.96 * np.sqrt(((p/100) * (1 - p/100)) / l_total) * 100 for p in l_pct]
    t_errors = [1.96 * np.sqrt(((p/100) * (1 - p/100)) / t_total) * 100 for p in t_pct]

    rects1 = ax.bar(x - width/2, l_pct, width, 
                    yerr=l_errors, capsize=5, error_kw={'elinewidth': 1.5, 'alpha': 0.7}, 
                    label=f'Learners (N={l_total})', color='#3498db', edgecolor='black')
    
    rects2 = ax.bar(x + width/2, t_pct, width, 
                    yerr=t_errors, capsize=5, error_kw={'elinewidth': 1.5, 'alpha': 0.7}, 
                    label=f'Teachers (N={t_total})', color='#e74c3c', edgecolor='black')

    ax.set_ylabel('Frequency (%)', fontsize=12)
    ax.set_title('Distribution of Pairwise Preferences: Learners vs. Teachers (with 95% Confidence Intervals)', fontsize=14, fontweight='bold', pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_labels, fontsize=12)
    ax.legend(fontsize=12)

    def autolabel(rects, errors):
        for rect, err in zip(rects, errors):
            height = rect.get_height()
            ax.annotate(f'{height:.1f}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height + err),
                        xytext=(0, 5),
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=10)

    autolabel(rects1, l_errors)
    autolabel(rects2, t_errors)

    sns.despine(left=True, bottom=False)
    plt.tight_layout()
    
    output_filename = "pairwise_preferences_distribution.png"
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    print(f"Figure saved at: {output_filename}")
    
    plt.show()

if __name__ == "__main__":
    l_counts, t_counts, l_total, t_total = load_preference_data()
    
    print("--- RESULTS ---")
    print(f"Learners (N={l_total}):", l_counts)
    print(f"Teachers (N={t_total}):", t_counts)
    
    test_tie_propensity(l_counts, t_counts)
    
    test_position_bias(l_counts, t_counts)
    
    plot_preferences_distribution(l_counts, t_counts, l_total, t_total)

In [ ]:
import json
import numpy as np
from scipy.stats import mannwhitneyu
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

# File path (adjust if necessary)
DATA_PATH = '../data/students_teacher_gold.json'

def analyze_ratings():
    # Dictionaries to store ratings per model (excluding ties)
    student_ratings_per_model = defaultdict(list)
    teacher_ratings_per_model = defaultdict(list)

    # Global lists for the statistical test (including all ratings, even ties)
    all_student_ratings = []
    all_teacher_ratings = []

    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            f.seek(0)
            data = [json.loads(line) for line in f]

    for item in data:
        # 1. Extract student ratings
        student_pref = item.get("preferred_model")
        student_rating = item.get("rating", 0)
        
        if student_rating > 0:
            # Add to the global pool (to reach the full N, including both_equal)
            all_student_ratings.append(student_rating)
            
            # Attribute to a specific model ONLY if there is a clear winner
            if student_pref not in ["both_equal", None]:
                student_ratings_per_model[student_pref].append(student_rating)
            
        # 2. Extract teacher ratings
        teacher_annotations = item.get("teacher_annotation_data", [])
        for teacher_anno in teacher_annotations:
            raw_teacher_pref = teacher_anno.get("preferred_model")
            teacher_rating = teacher_anno.get("rating", 0)
            
            # Translate "a" or "b" to the actual model name
            teacher_pref = None
            if raw_teacher_pref in ["a", "A", "model_a", "Modèle A"]:
                teacher_pref = item.get("model_a_name")
            elif raw_teacher_pref in ["b", "B", "model_b", "Modèle B"]:
                teacher_pref = item.get("model_b_name")
            else:
                teacher_pref = raw_teacher_pref # Handles "both_equal" or pre-resolved names
            
            if teacher_rating > 0:
                # Add to the global pool (including both_equal)
                all_teacher_ratings.append(teacher_rating)
                
                # Attribute to a specific model ONLY if there is a clear winner
                if teacher_pref not in ["both_equal", None]:
                    teacher_ratings_per_model[teacher_pref].append(teacher_rating)

    print("=== AVERAGE RATINGS PER MODEL ===")
    all_models = sorted(list(set(student_ratings_per_model.keys()) | set(teacher_ratings_per_model.keys())))
    
    for model in all_models:
        s_ratings = student_ratings_per_model.get(model, [])
        t_ratings = teacher_ratings_per_model.get(model, [])
        
        s_mean = np.mean(s_ratings) if s_ratings else 0.0
        t_mean = np.mean(t_ratings) if t_ratings else 0.0
        
        print(f"Model: {model}")
        print(f"  Students : {s_mean:.2f}/5 (N={len(s_ratings)})")
        print(f"  Teachers : {t_mean:.2f}/5 (N={len(t_ratings)})\n")

    print("=== GLOBAL COMPARISON: STUDENTS VS TEACHERS ===")
    if not all_student_ratings or not all_teacher_ratings:
        print("Insufficient data to perform the statistical test.")
        return

    s_global_mean = np.mean(all_student_ratings)
    t_global_mean = np.mean(all_teacher_ratings)
    
    print(f"Global Mean (Students) : {s_global_mean:.2f}/5 (N={len(all_student_ratings)})")
    print(f"Global Mean (Teachers) : {t_global_mean:.2f}/5 (N={len(all_teacher_ratings)})")

    stat, p_value = mannwhitneyu(all_student_ratings, all_teacher_ratings, alternative='two-sided')
    
    print(f"Mann-Whitney U Test Results:")
    print(f"U Statistic = {stat}")
    print(f"p-value = {p_value:.4f}")

    plot_rating_distributions(all_student_ratings, all_teacher_ratings)
    plot_ratings_per_model(student_ratings_per_model, teacher_ratings_per_model, all_models)

def plot_rating_distributions(student_ratings, teacher_ratings):
    """
    Generates a grouped bar chart showing the frequency of each rating (1 to 5)
    for both students and teachers, with 95% Confidence Intervals.
    """
    sns.set_theme(style="whitegrid")
    ratings_range = [1, 2, 3, 4, 5]
    
    n_s = len(student_ratings)
    n_t = len(teacher_ratings)
    
    s_counts = [student_ratings.count(r) for r in ratings_range]
    t_counts = [teacher_ratings.count(r) for r in ratings_range]
    
    print(f"Standard deviation (Students): {np.std(student_ratings, ddof=1):.2f}")
    print(f"Standard deviation (Teachers): {np.std(teacher_ratings, ddof=1):.2f}")
    
    s_percentages = [count / n_s * 100 for count in s_counts]
    t_percentages = [count / n_t * 100 for count in t_counts]

    # 95% Confidence Intervals for proportions
    s_errors = [1.96 * np.sqrt(((p/100) * (1 - p/100)) / n_s) * 100 for p in s_percentages]
    t_errors = [1.96 * np.sqrt(((p/100) * (1 - p/100)) / n_t) * 100 for p in t_percentages]

    x = np.arange(len(ratings_range))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, 6))
    
    bars1 = ax.bar(x - width/2, s_percentages, width, 
                   yerr=s_errors, capsize=4, error_kw={'elinewidth': 1.2, 'alpha': 0.7},
                   label=f'Learners (N={n_s})', color='#3498db', edgecolor='black')
    
    bars2 = ax.bar(x + width/2, t_percentages, width, 
                   yerr=t_errors, capsize=4, error_kw={'elinewidth': 1.2, 'alpha': 0.7},
                   label=f'Teachers (N={n_t})', color='#e74c3c', edgecolor='black')

    ax.set_ylabel('Frequency (%)', fontsize=12)
    ax.set_xlabel('Pointwise Rating', fontsize=12)
    ax.set_title('Distribution of Winning Pointwise Ratings: Learners vs. Teachers (with 95% Confidence Intervals)', fontsize=13, pad=15, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(ratings_range, fontsize=11)
    ax.set_ylim(0, max(s_percentages + t_percentages) + 18)
    ax.legend(fontsize=11)

    # Annotations textuelles au-dessus des moustaches
    for bars, errors in [(bars1, s_errors), (bars2, t_errors)]:
        for bar, err in zip(bars, errors):
            height = bar.get_height()
            if height > 0:
                ax.annotate(f'{height:.1f}%',
                            xy=(bar.get_x() + bar.get_width() / 2, height + err),
                            xytext=(0, 4),
                            textcoords="offset points",
                            ha='center', va='bottom', fontsize=9)

    sns.despine(left=True, bottom=False)
    plt.tight_layout()
    plt.savefig("ratings_distribution_plot.png", dpi=300)
    print("\n-> Global distribution plot successfully saved as 'ratings_distribution_plot.png'")
    plt.show()

def plot_ratings_per_model(student_dict, teacher_dict, all_models):
    """
    Generates a horizontal bar chart showing the average rating per model,
    comparing students and teachers, including the sample size (N) for each bar.
    """
    sns.set_theme(style="whitegrid")
    
    # Calculate means and counts (default to 0 if no ratings for that group)
    s_means = [np.mean(student_dict[m]) if m in student_dict and student_dict[m] else 0 for m in all_models]
    t_means = [np.mean(teacher_dict[m]) if m in teacher_dict and teacher_dict[m] else 0 for m in all_models]
    
    s_counts = [len(student_dict[m]) if m in student_dict else 0 for m in all_models]
    t_counts = [len(teacher_dict[m]) if m in teacher_dict else 0 for m in all_models]
    
    y = np.arange(len(all_models))
    height = 0.35

    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Horizontal bars
    bars1 = ax.barh(y - height/2, s_means, height, label='Learners (N=44)', color='#3498db')
    bars2 = ax.barh(y + height/2, t_means, height, label='Teachers (N=45)', color='#e74c3c')

    ax.set_xlabel('Average Rating (out of 5)', fontsize=12)
    ax.set_title('Average Winning Pointwise Rating per Model: Learners vs. Teachers', fontsize=14, pad=15)
    ax.set_yticks(y)
    
    # Clean up model names for the Y-axis
    clean_models = [m.split('/')[-1] if '/' in m else m for m in all_models]
    ax.set_yticklabels(clean_models, fontsize=10)
    
    # Extended X-axis limit to leave room for the text annotations (e.g. "5.00 (N=12)")
    ax.set_xlim(1, 5.8) 
    ax.legend()

    # Add text annotations on the bars for Learners
    for i, bar in enumerate(bars1):
        width = bar.get_width()
        count = s_counts[i]
        if width > 0:
            ax.annotate(f'{width:.2f} (N={count})',
                        xy=(width, bar.get_y() + bar.get_height() / 2),
                        xytext=(5, 0), # 5 points horizontal offset
                        textcoords="offset points",
                        ha='left', va='center', fontsize=9)

    # Add text annotations on the bars for Teachers
    for i, bar in enumerate(bars2):
        width = bar.get_width()
        count = t_counts[i]
        if width > 0:
            ax.annotate(f'{width:.2f} (N={count})',
                        xy=(width, bar.get_y() + bar.get_height() / 2),
                        xytext=(5, 0),
                        textcoords="offset points",
                        ha='left', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig("ratings_per_model_plot.png", dpi=300)
    print("-> Per-model ratings plot successfully saved as 'ratings_per_model_plot.png'")
    plt.show()

analyze_ratings()

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, binomtest

LEARNERS_PATH = "../data/reactions.json"
TEACHERS_PATH = "../data/students_teacher_gold.json"

LABELS = ["complete", "correct", "relevant", "concise", "scaffolding", "understandable"]

def load_data():
    
    with open(LEARNERS_PATH, 'r', encoding='utf-8') as f:
        learners_data = json.load(f)
        
    learner_valid_items = [
        x["labels"] for x in learners_data 
        if x.get("labels") is not None and any(x["labels"].values())
    ]
    
    learner_total = len(learner_valid_items)
    learner_counts = {label: sum(1 for item in learner_valid_items if item.get(label) is True) for label in LABELS}
    learner_pct = {k: (v / learner_total) * 100 for k, v in learner_counts.items()}

    with open(TEACHERS_PATH, 'r', encoding='utf-8') as f:
        gold_data = json.load(f)
        
    teachers_data = [ann for item in gold_data for ann in item.get("teacher_annotation_data", [])]
    teacher_valid_items = [
        x for x in teachers_data 
        if any(x.get(label, False) for label in LABELS)
    ]
    
    teacher_total = len(teacher_valid_items)
    teacher_counts = {label: sum(1 for item in teacher_valid_items if item.get(label) is True) for label in LABELS}
    teacher_pct = {k: (v / teacher_total) * 100 for k, v in teacher_counts.items()}

    return learner_counts, teacher_counts, learner_pct, teacher_pct, learner_total, teacher_total


def run_statistical_tests(l_counts, t_counts, l_total, t_total):
    print("="*60)
    print("STATISTICAL ANALYSIS: LEARNERS VS TEACHERS")
    print("="*60)
    print("1. CHI-SQUARE TESTS (Individual Labels)")
    print("-" * 60)
    
    teachers_lower_count = 0
    
    for label in LABELS:
        l_true = l_counts[label]
        l_false = l_total - l_true
        t_true = t_counts[label]
        t_false = t_total - t_true
        
        table = [[l_true, l_false], [t_true, t_false]]
        
        chi2, p, dof, expected = chi2_contingency(table)
        
        l_pct = (l_true / l_total) * 100
        t_pct = (t_true / t_total) * 100
        if t_pct < l_pct:
            teachers_lower_count += 1
            
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        print(f"{label.capitalize():<15} | Learners: {l_pct:04.1f}% | Teachers: {t_pct:04.1f}% | p-value: {p:.5f} {sig}")

    print("2. BINOMIAL TEST (Global Asymmetry)")
    print("-" * 60)
    binom_res = binomtest(teachers_lower_count, len(LABELS), 0.5, alternative='two-sided')
    p_binom = binom_res.pvalue
    
    print(f"Teachers scored lower than learners on {teachers_lower_count}/{len(LABELS)} labels.")
    
    sig_binom = "***" if p_binom < 0.001 else "**" if p_binom < 0.01 else "*" if p_binom < 0.05 else "ns"
    print(f"Binomial test p-value: {p_binom:.5f} {sig_binom}")
    print("="*60)


def plot_distributions_with_ci(learner_pct, teacher_pct, n_learners, n_teachers):
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(12, 6))

    x = np.arange(len(LABELS))
    width = 0.35

    l_errors = [1.96 * np.sqrt(((learner_pct[l]/100) * (1 - learner_pct[l]/100)) / n_learners) * 100 for l in LABELS]
    t_errors = [1.96 * np.sqrt(((teacher_pct[l]/100) * (1 - teacher_pct[l]/100)) / n_teachers) * 100 for l in LABELS]

    rects1 = ax.bar(x - width/2, [learner_pct[l] for l in LABELS], width, 
                    yerr=l_errors, capsize=5, error_kw={'elinewidth': 1.5, 'alpha': 0.7}, 
                    label='Learners (N=37)', color='#3498db', edgecolor='black')
    
    rects2 = ax.bar(x + width/2, [teacher_pct[l] for l in LABELS], width, 
                    yerr=t_errors, capsize=5, error_kw={'elinewidth': 1.5, 'alpha': 0.7}, 
                    label='Teachers (N=41)', color='#e74c3c', edgecolor='black')
    
    ax.set_ylabel('Selection Frequency (%)', fontsize=12)
    ax.set_title('Label Distribution for Preferred Model Outputs: Learners vs. Teachers (with 95% Confidence Intervals)', fontsize=14, fontweight='bold', pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels([l.capitalize() for l in LABELS], fontsize=12)
    ax.legend(fontsize=12)

    # Print '%' above the bars
    def autolabel(rects, errors):
        for rect, err in zip(rects, errors):
            height = rect.get_height()
            ax.annotate(f'{height:.1f}%',
                        xy=(rect.get_x() + rect.get_width() / 2, height + err),
                        xytext=(0, 5),
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=10)

    autolabel(rects1, l_errors)
    autolabel(rects2, t_errors)

    sns.despine(left=True, bottom=False)
    plt.tight_layout()
    
    output_filename = "label_contingency_plot_with_CI.png"
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    print(f"\\nGraphique sauvegardé avec succès sous '{output_filename}'")
    plt.show()

if __name__ == "__main__":
    l_counts, t_counts, l_pct, t_pct, l_total, t_total = load_data()
    run_statistical_tests(l_counts, t_counts, l_total, t_total)    
    plot_distributions_with_ci(l_pct, t_pct, l_total, t_total)

## RQ2

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy import stats
from tqdm import tqdm

INPUT_DATA_PATH = os.path.join("../data", "students_teacher_gold.json")

def resolve_choice(raw_choice, model_a: str, model_b: str) -> str | None:
    """Traduit la valeur brute en 'a', 'b', ou 'both_equal'."""
    if not raw_choice:
        return None
    c = str(raw_choice).strip().lower()
    if c in ['none', '', 'null', 'nan']:
        return None
    if c in ['a', 'model_a', 'model a']:
        return 'a'
    if c in ['b', 'model_b', 'model b']:
        return 'b'
    if c in ['both_equal', 'tie', 'both', 'equal', 'both equal', 'both_are_equal']:
        return 'both_equal'
    
    ma_l = str(model_a).strip().lower()
    mb_l = str(model_b).strip().lower()
    
    if ma_l and (c == ma_l or c.endswith(ma_l) or ma_l.endswith(c)):
        return 'a'
    if mb_l and (c == mb_l or c.endswith(mb_l) or mb_l.endswith(c)):
        return 'b'
        
    return None

def load_paired_dialogue_items(input_data_path: str = INPUT_DATA_PATH) -> list[dict]:
    with open(input_data_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    extracted_items = []
    for item in data:
        model_a = item.get("model_a_name", "")
        model_b = item.get("model_b_name", "")
        if not model_a or not model_b:
            continue
            
        teacher_choices = []
        for ann in item.get("teacher_annotation_data", []):
            if isinstance(ann, dict):
                c = resolve_choice(ann.get("preferred_model"), model_a, model_b)
                if c:
                    teacher_choices.append(c)
                
        student_choices = []
        student_pref = item.get("preferred_model")
        c = resolve_choice(student_pref, model_a, model_b)
        if c:
            student_choices.append(c)
                
        extracted_items.append({
            "model_a": model_a,
            "model_b": model_b,
            "teachers": teacher_choices,
            "learners": student_choices
        })
    return extracted_items

def fit_btd_map(
    pairs: list[tuple[str, str, str]], 
    all_models: list[str], 
    l2_reg: float = 0.05
) -> tuple[dict[str, float], float]:
    n_models = len(all_models)
    model_to_idx = {m: i for i, m in enumerate(all_models)}
    
    wins = np.zeros((n_models, n_models), dtype=float)
    ties = np.zeros((n_models, n_models), dtype=float)
    
    for ma, mb, choice in pairs:
        if ma not in model_to_idx or mb not in model_to_idx:
            continue
        ia, ib = model_to_idx[ma], model_to_idx[mb]
        if choice == 'a':
            wins[ia, ib] += 1.0
        elif choice == 'b':
            wins[ib, ia] += 1.0
        elif choice == 'both_equal':
            ties[ia, ib] += 1.0
            ties[ib, ia] += 1.0

    def btd_loss(params):
        thetas = params[:-1]
        nu = params[-1]
        if nu <= 1e-5:
            return 1e9

        nll = 0.0
        for i in range(n_models):
            for j in range(i + 1, n_models):
                w_ij, w_ji, t_ij = wins[i, j], wins[j, i], ties[i, j]
                if w_ij == 0 and w_ji == 0 and t_ij == 0:
                    continue

                exp_i, exp_j = np.exp(thetas[i]), np.exp(thetas[j])
                denom = exp_i + exp_j + nu * np.sqrt(exp_i * exp_j)
                if denom <= 0:
                    return 1e9

                nll -= (w_ij * np.log(exp_i / denom) +
                        w_ji * np.log(exp_j / denom) +
                        t_ij * np.log((nu * np.sqrt(exp_i * exp_j)) / denom))
        
        penalty = 0.5 * l2_reg * np.sum(thetas ** 2)
        return nll + penalty

    init_params = np.zeros(n_models + 1)
    init_params[-1] = 1.0
    bounds = [(None, None)] * n_models + [(1e-4, None)]

    with np.errstate(all='ignore'):
        opt = minimize(btd_loss, init_params, method='L-BFGS-B', bounds=bounds)

    thetas = opt.x[:-1]
    thetas -= np.mean(thetas)
    nu = float(max(opt.x[-1], 1e-4))

    scores = {m: float(thetas[i]) for i, m in enumerate(all_models)}
    return scores, nu

def compute_ewr_and_es(scores: dict[str, float], nu: float) -> tuple[dict[str, float], dict[str, float]]:
    models = list(scores.keys())
    n_models = len(models)
    ewr_dict = {}
    es_dict = {}
    
    for ma in models:
        th_a = scores[ma]
        sum_p_wins = 0.0
        sum_p_scores = 0.0
        
        for mb in models:
            if ma == mb:
                continue
            th_b = scores[mb]
            denom = np.exp(th_a) + np.exp(th_b) + nu * np.sqrt(np.exp(th_a) * np.exp(th_b))
            p_win = np.exp(th_a) / denom
            p_tie = (nu * np.sqrt(np.exp(th_a) * np.exp(th_b))) / denom
            
            sum_p_wins += p_win
            sum_p_scores += (p_win + 0.5 * p_tie)
            
        ewr_dict[ma] = float(sum_p_wins / (n_models - 1)) if n_models > 1 else 0.0
        es_dict[ma] = float(sum_p_scores / (n_models - 1)) if n_models > 1 else 0.0
        
    return ewr_dict, es_dict


def safe_ci_pct(arr: list[float], low: float = 2.5, high: float = 97.5) -> str:
    valid = [x for x in arr if not np.isnan(x)]
    return f"[{np.percentile(valid, low):.1%}, {np.percentile(valid, high):.1%}]" if valid else "N/A"

def safe_ci_val(arr: list[float], low: float = 2.5, high: float = 97.5) -> list[float]:
    valid = [x for x in arr if not np.isnan(x)]
    return [float(np.percentile(valid, low)), float(np.percentile(valid, high))] if valid else [np.nan, np.nan]

def bootstrap_cohort_btd_full(items: list[dict], n_iterations: int = 1000, seed: int = 2026) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(seed)
    n_items = len(items)
    all_models = sorted(list(set([it["model_a"] for it in items] + [it["model_b"] for it in items])))

    obs_t_pairs = [(it["model_a"], it["model_b"], c) for it in items for c in it["teachers"]]
    obs_l_pairs = [(it["model_a"], it["model_b"], c) for it in items for c in it["learners"]]

    print(f"--- LOADING DIAGNOSTIC ---")
    print(f"Processed dialogues        : {n_items}")
    print(f"Extracted teacher votes    : {len(obs_t_pairs)}")
    print(f"Extracted learner votes    : {len(obs_l_pairs)}")
    print("--------------------------------")
    
    if len(obs_l_pairs) == 0:
        raise ValueError("No learner votes found in the dataset. Check JSON data.")

    scores_t, nu_t = fit_btd_map(obs_t_pairs, all_models)
    scores_l, nu_l = fit_btd_map(obs_l_pairs, all_models)
    
    ewr_t, es_t = compute_ewr_and_es(scores_t, nu_t)
    ewr_l, es_l = compute_ewr_and_es(scores_l, nu_l)

    boot_rho, boot_tau = [], []
    boot_theta_l = {m: [] for m in all_models}
    boot_theta_t = {m: [] for m in all_models}
    boot_ewr_l = {m: [] for m in all_models} 
    boot_ewr_t = {m: [] for m in all_models}
    boot_es_l = {m: [] for m in all_models}
    boot_es_t = {m: [] for m in all_models}

    for _ in tqdm(range(n_iterations), desc="Bootstrapping BTD Metrics"):
        idx = rng.choice(n_items, size=n_items, replace=True)
        resampled = [items[i] for i in idx]

        t_pairs = [(it["model_a"], it["model_b"], c) for it in resampled for c in it["teachers"]]
        l_pairs = [(it["model_a"], it["model_b"], c) for it in resampled for c in it["learners"]]

        try:
            b_sc_t, b_nu_t = fit_btd_map(t_pairs, all_models)
            b_sc_l, b_nu_l = fit_btd_map(l_pairs, all_models)
            
            b_ewr_t, b_es_t = compute_ewr_and_es(b_sc_t, b_nu_t)
            b_ewr_l, b_es_l = compute_ewr_and_es(b_sc_l, b_nu_l)

            b_vec_t = [b_sc_t[m] for m in all_models]
            b_vec_l = [b_sc_l[m] for m in all_models]

            if np.std(b_vec_l) > 1e-5 and np.std(b_vec_t) > 1e-5:
                r = stats.spearmanr(b_vec_l, b_vec_t).statistic
                k = stats.kendalltau(b_vec_l, b_vec_t).statistic
                if not np.isnan(r) and not np.isnan(k):
                    boot_rho.append(r)
                    boot_tau.append(k)
                    
            for m in all_models:
                boot_theta_t[m].append(b_sc_t[m])
                boot_theta_l[m].append(b_sc_l[m])
                boot_ewr_t[m].append(b_ewr_t[m])
                boot_ewr_l[m].append(b_ewr_l[m])
                boot_es_t[m].append(b_es_t[m])
                boot_es_l[m].append(b_es_l[m])
        except Exception:
            continue

    leaderboard_df = pd.DataFrame({
        "Model": all_models,
        # Learners
        "Theta_Learners": [scores_l[m] for m in all_models],
        "Theta_Learners_CI_low": [safe_ci_val(boot_theta_l[m])[0] for m in all_models],
        "Theta_Learners_CI_high": [safe_ci_val(boot_theta_l[m])[1] for m in all_models],
        "EWR_Learners": [ewr_l[m] for m in all_models],
        "EWR_Learners_CI_low": [safe_ci_val(boot_ewr_l[m])[0] for m in all_models],
        "EWR_Learners_CI_high": [safe_ci_val(boot_ewr_l[m])[1] for m in all_models],
        "ES_Learners": [es_l[m] for m in all_models],
        "ES_Learners_CI_low": [safe_ci_val(boot_es_l[m])[0] for m in all_models],
        "ES_Learners_CI_high": [safe_ci_val(boot_es_l[m])[1] for m in all_models],
        
        # Teachers
        "Theta_Teachers": [scores_t[m] for m in all_models],
        "Theta_Teachers_CI_low": [safe_ci_val(boot_theta_t[m])[0] for m in all_models],
        "Theta_Teachers_CI_high": [safe_ci_val(boot_theta_t[m])[1] for m in all_models],
        "EWR_Teachers": [ewr_t[m] for m in all_models],
        "EWR_Teachers_CI_low": [safe_ci_val(boot_ewr_t[m])[0] for m in all_models],
        "EWR_Teachers_CI_high": [safe_ci_val(boot_ewr_t[m])[1] for m in all_models],
        "ES_Teachers": [es_t[m] for m in all_models],
        "ES_Teachers_CI_low": [safe_ci_val(boot_es_t[m])[0] for m in all_models],
        "ES_Teachers_CI_high": [safe_ci_val(boot_es_t[m])[1] for m in all_models],
    }).sort_values(by="ES_Teachers", ascending=False).reset_index(drop=True)

    vec_l_obs = [scores_l[m] for m in all_models]
    vec_t_obs = [scores_t[m] for m in all_models]
    
    obs_rho = stats.spearmanr(vec_l_obs, vec_t_obs).statistic if np.std(vec_l_obs) > 1e-5 else np.nan
    obs_tau = stats.kendalltau(vec_l_obs, vec_t_obs).statistic if np.std(vec_l_obs) > 1e-5 else np.nan

    summary_stats = {
        "nu_teachers": nu_t,
        "nu_learners": nu_l,
        "spearman_rho_obs": obs_rho,
        "spearman_rho_mean": np.mean(boot_rho) if len(boot_rho) > 0 else np.nan,
        "spearman_rho_ci": safe_ci_val(boot_rho),
        "kendall_tau_obs": obs_tau,
        "kendall_tau_mean": np.mean(boot_tau) if len(boot_tau) > 0 else np.nan,
        "kendall_tau_ci": safe_ci_val(boot_tau),
        "boot_rho_samples": boot_rho,
        "boot_tau_samples": boot_tau
    }

    return leaderboard_df, summary_stats


def visualize_btd_results(leaderboard_df: pd.DataFrame, summary_stats: dict):
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    df_sorted = leaderboard_df.sort_values(by="Theta_Teachers", ascending=True)
    y_pos = np.arange(len(df_sorted))
    
    # Rankings

    axes[0].hlines(y=y_pos, xmin=df_sorted["Theta_Learners"], xmax=df_sorted["Theta_Teachers"], 
                   color="grey", alpha=0.4, linewidth=2.5, zorder=1)
    
    err_theta_l = [
        df_sorted["Theta_Learners"] - df_sorted["Theta_Learners_CI_low"],
        df_sorted["Theta_Learners_CI_high"] - df_sorted["Theta_Learners"]
    ]
    err_theta_t = [
        df_sorted["Theta_Teachers"] - df_sorted["Theta_Teachers_CI_low"],
        df_sorted["Theta_Teachers_CI_high"] - df_sorted["Theta_Teachers"]
    ]
    
    axes[0].errorbar(df_sorted["Theta_Learners"], y_pos, xerr=err_theta_l, fmt='none', ecolor="#1f77b4", alpha=0.6, capsize=3, elinewidth=1.5, zorder=2)
    axes[0].errorbar(df_sorted["Theta_Teachers"], y_pos, xerr=err_theta_t, fmt='none', ecolor="#d62728", alpha=0.6, capsize=3, elinewidth=1.5, zorder=2)
    
    axes[0].scatter(df_sorted["Theta_Learners"], y_pos, color="#1f77b4", s=90, label="Learners ($\hat{\\theta}$)", zorder=3)
    axes[0].scatter(df_sorted["Theta_Teachers"], y_pos, color="#d62728", s=90, label="Teachers ($\hat{\\theta}$)", zorder=3)
    
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels(df_sorted["Model"], fontsize=10)
    axes[0].axvline(0, color="black", linestyle="--", alpha=0.5)
    axes[0].set_xlabel("Latent Strength Parameter ($\hat{\\theta}$, Mean-Centered)", fontsize=11)
    axes[0].set_title("A. Latent Strength ($\hat{\\theta}$) with 95% CIs", fontsize=12, fontweight="bold")
    axes[0].legend(loc="lower right", frameon=True)

    # Expected Score and Expected Win Rate
    
    bar_width = 0.35
    
    err_es_l = [
        (df_sorted["ES_Learners"] - df_sorted["ES_Learners_CI_low"]) * 100,
        (df_sorted["ES_Learners_CI_high"] - df_sorted["ES_Learners"]) * 100
    ]
    err_es_t = [
        (df_sorted["ES_Teachers"] - df_sorted["ES_Teachers_CI_low"]) * 100,
        (df_sorted["ES_Teachers_CI_high"] - df_sorted["ES_Teachers"]) * 100
    ]
    
    axes[1].barh(y_pos - bar_width/2, df_sorted["ES_Learners"] * 100, xerr=err_es_l, height=bar_width, 
                 color="#3498db", alpha=0.5, label="Learners (ES)", capsize=3, error_kw={'alpha': 0.6, 'lw': 1.5}, zorder=2)
    axes[1].barh(y_pos + bar_width/2, df_sorted["ES_Teachers"] * 100, xerr=err_es_t, height=bar_width, 
                 color="#e74c3c", alpha=0.5, label="Teachers (ES)", capsize=3, error_kw={'alpha': 0.6, 'lw': 1.5}, zorder=2)
    
    err_ewr_l = [
        (df_sorted["EWR_Learners"] - df_sorted["EWR_Learners_CI_low"]) * 100,
        (df_sorted["EWR_Learners_CI_high"] - df_sorted["EWR_Learners"]) * 100
    ]
    err_ewr_t = [
        (df_sorted["EWR_Teachers"] - df_sorted["EWR_Teachers_CI_low"]) * 100,
        (df_sorted["EWR_Teachers_CI_high"] - df_sorted["EWR_Teachers"]) * 100
    ]
    
    eb_l = axes[1].errorbar(df_sorted["EWR_Learners"] * 100, y_pos - bar_width/2, xerr=err_ewr_l, 
                            fmt='D', markerfacecolor='white', markeredgecolor='#2980b9', 
                            ecolor='#2980b9', elinewidth=1.5, capsize=0, zorder=4, 
                            label="Learners (EWR)")
    eb_l[2][0].set_linestyle('--')
    
    eb_t = axes[1].errorbar(df_sorted["EWR_Teachers"] * 100, y_pos + bar_width/2, xerr=err_ewr_t, 
                            fmt='D', markerfacecolor='white', markeredgecolor='#c0392b', 
                            ecolor='#c0392b', elinewidth=1.5, capsize=0, zorder=4, 
                            label="Teachers (EWR)")
    eb_t[2][0].set_linestyle('--')
    
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels([])
    axes[1].axvline(50, color="black", linestyle="--", alpha=0.4, label="Baseline (50%)")
    axes[1].set_xlabel("Expected Win Rate (EWR) & Score Rate (ES) in %", fontsize=11)
    axes[1].set_title("B. ES (Bars) vs EWR (Diamonds) with 95% CIs", fontsize=12, fontweight="bold")
    axes[1].legend(loc="lower right", frameon=True, ncol=2, fontsize=9)

    plt.tight_layout()
    plt.show()

    # Rho + Tau

    plt.figure(figsize=(10, 5))
    rho_obs = summary_stats["spearman_rho_obs"]
    rho_ci = summary_stats["spearman_rho_ci"]
    tau_obs = summary_stats["kendall_tau_obs"]
    tau_ci = summary_stats["kendall_tau_ci"]

    sns.kdeplot(summary_stats["boot_rho_samples"], fill=True, color="#2ca02c", alpha=0.4,
                label=f"Spearman's $\\rho$ (Observed = {rho_obs:.2f}, 95% CI: [{rho_ci[0]:.2f}, {rho_ci[1]:.2f}])")
    sns.kdeplot(summary_stats["boot_tau_samples"], fill=True, color="#9467bd", alpha=0.4,
                label=f"Kendall's $\\tau$ (Observed = {tau_obs:.2f}, 95% CI: [{tau_ci[0]:.2f}, {tau_ci[1]:.2f}])")

    plt.axvline(0, color="red", linestyle="--", linewidth=1.5, label="Null Correlation (0.0)")
    if not np.isnan(rho_obs):
        plt.axvline(rho_obs, color="#2ca02c", linestyle=":", linewidth=2)
    if not np.isnan(tau_obs):
        plt.axvline(tau_obs, color="#9467bd", linestyle=":", linewidth=2)

    plt.title("Bootstrapped Inter-Role Rank Alignment (N=1000 Iterations)", fontsize=12, fontweight="bold", pad=12)
    plt.xlabel("Rank Correlation Coefficient", fontsize=11)
    plt.ylabel("Bootstrap Density", fontsize=11)
    plt.legend(loc="upper left", frameon=True, fontsize=10)
    plt.tight_layout()
    plt.show()


items = load_paired_dialogue_items(INPUT_DATA_PATH)
leaderboard_df, summary_stats = bootstrap_cohort_btd_full(items, n_iterations=1000, seed=2026)

print("=== TIE PARAMETERS ===")
print(f"Nu Teachers : {summary_stats['nu_teachers']:.3f} | Nu Learners : {summary_stats['nu_learners']:.3f}")

print("=== FULL LEADERBOARD ===")
def format_ci(val, low, high):
    return f"{val:.1%} [{low:.1%}, {high:.1%}]"

disp_df = leaderboard_df.copy()
disp_df["EWR_Learners_Disp"] = disp_df.apply(lambda row: format_ci(row["EWR_Learners"], row["EWR_Learners_CI_low"], row["EWR_Learners_CI_high"]), axis=1)
disp_df["ES_Learners_Disp"] = disp_df.apply(lambda row: format_ci(row["ES_Learners"], row["ES_Learners_CI_low"], row["ES_Learners_CI_high"]), axis=1)
disp_df["EWR_Teachers_Disp"] = disp_df.apply(lambda row: format_ci(row["EWR_Teachers"], row["EWR_Teachers_CI_low"], row["EWR_Teachers_CI_high"]), axis=1)
disp_df["ES_Teachers_Disp"] = disp_df.apply(lambda row: format_ci(row["ES_Teachers"], row["ES_Teachers_CI_low"], row["ES_Teachers_CI_high"]), axis=1)

print(disp_df[[
    "Model", "Theta_Learners", "EWR_Learners_Disp", "ES_Learners_Disp", 
    "Theta_Teachers", "EWR_Teachers_Disp", "ES_Teachers_Disp"
]].to_string(index=False))

print("\n=== RANK CORRELATIONS ===")
ci_rho = summary_stats['spearman_rho_ci']
ci_tau = summary_stats['kendall_tau_ci']
print(f"Spearman's Rho : Obs = {summary_stats['spearman_rho_obs']:.3f} | Mean = {summary_stats['spearman_rho_mean']:.3f} | 95% CI = [{ci_rho[0]:.3f}, {ci_rho[1]:.3f}]")
print(f"Kendall's Tau  : Obs = {summary_stats['kendall_tau_obs']:.3f} | Mean = {summary_stats['kendall_tau_mean']:.3f} | 95% CI = [{ci_tau[0]:.3f}, {ci_tau[1]:.3f}]")

visualize_btd_results(leaderboard_df, summary_stats)

## RQ3

In [ ]:
import json
import pandas as pd

files = {
    "Phi-4": "../data/judge_runs/reactions__phi4.jsonl",
    "Qwen3-235B": "../data/judge_runs/reactions__qwen3-235b.jsonl"
}

def analyze_positional_stability_fixed(filepath, judge_name):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
            
    df = pd.DataFrame(data)
    
    # Separate normal and swapped
    df_normal = df[df['is_swapped'] == False].copy()
    df_swapped = df[df['is_swapped'] == True].copy()
    
    # Merge on reaction_id and annotator_id to compare the same test under two angles
    merged = pd.merge(
        df_normal, 
        df_swapped, 
        on=['reaction_id', 'annotator_id'],
        suffixes=('_normal', '_swapped')
    )
    
    if len(merged) == 0:
        print(f"[{judge_name}] No paired comparisons found.")
        return

    consistent_count = 0
    total_count = len(merged)
    
    for _, row in merged.iterrows():
        pref_norm = row['judge_preferred_model_normal']
        pref_swap = row['judge_preferred_model_swapped']
        
        if pref_norm == pref_swap:
            consistent_count += 1
            
    consistency_rate = (consistent_count / total_count) * 100 if total_count > 0 else 0
    
    print(f"==========================================")
    print(f"LLM Judge: {judge_name}")
    print(f"Evaluated pairs: {total_count}")
    print(f"Actual consistency rate: {consistency_rate:.2f}%")
    print(f"==========================================\n")

for name, path in files.items():
    analyze_positional_stability_fixed(path, name)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_verbosity_btd_multiple_judges(btd_judges_dict: dict, dataset_path: str = "synthetic_outputs.jsonl"):
    """
    Compute and overlay on the same graph the correlations between the average length
    of responses (completion tokens) and the BTD latent strength scores (theta)
    for multiple LLM judge score dictionaries.
    
    Parameters:
        btd_judges_dict (dict): Dictionary of dictionary of scores, ex: 
                                ```
                                {"phi-4": llm_phi4_reactions_scores, "qwen3-235b": llm_qwen3_235b_reactions_scores}
                                ```
        dataset_path (str): Path to the JSONL file containing the synthetic outputs.
    """
    if not os.path.exists(dataset_path):
        alt_path = os.path.join("..", dataset_path)
        if os.path.exists(alt_path):
            dataset_path = alt_path
        else:
            raise FileNotFoundError(f"Impossible de trouver le fichier dataset à : {dataset_path}")

    data = []
    with open(dataset_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    df = pd.DataFrame(data)
    
    tokens_dict = {}
    counts_dict = {}
    
    for _, row in df.iterrows():
        for side in ['model_a', 'model_b']:
            m_info = row.get(side, {})
            m_name = m_info.get('model_name')
            usage = m_info.get('llm_usage')
            if m_name and usage:
                tokens = usage.get('completion_tokens', 0)
                tokens_dict[m_name] = tokens_dict.get(m_name, 0) + tokens
                counts_dict[m_name] = counts_dict.get(m_name, 0) + 1
                
    avg_tokens = {m: tokens_dict[m] / counts_dict[m] for m in tokens_dict}
    
    plt.figure(figsize=(10, 6))
    palette = sns.color_palette("tab10", len(btd_judges_dict))
    
    results_summary = {}

    for idx, (judge_name, btd_scores) in enumerate(btd_judges_dict.items()):
        models = []
        thetas = []
        lengths = []
        
        for model, score_data in btd_scores.items():
            if isinstance(score_data, dict):
                theta = score_data.get('theta', score_data.get('score', None))
                if theta is None and 'btd_score' in score_data:
                    theta = score_data['btd_score']
            else:
                theta = float(score_data)
                
            if theta is not None:
                matched_key = None
                if model in avg_tokens:
                    matched_key = model
                else:
                    for k in avg_tokens.keys():
                        if model in k or k in model:
                            matched_key = k
                            break
                            
                if matched_key:
                    models.append(model)
                    thetas.append(theta)
                    lengths.append(avg_tokens[matched_key])
                    
        if len(thetas) < 3:
            print(f"[Warning] Not enough data points ({len(thetas)}) for the judge {judge_name}.")
            continue

        spearman_corr, spearman_p = stats.spearmanr(lengths, thetas)
        pearson_corr, pearson_p = stats.pearsonr(lengths, thetas)
        
        results_summary[judge_name] = {
            "spearman_rho": spearman_corr,
            "spearman_p": spearman_p,
            "pearson_r": pearson_corr,
            "pearson_p": pearson_p
        }
        
        print("-" * 60)
        print(f" Verbosity Bias Analysis: BTD vs Response Length ({judge_name}) ".center(60, ' '))
        print(f"Spearman's Rho : {spearman_corr:.3f} (p-value = {spearman_p:.4f})")
        print(f"Pearson's r    : {pearson_corr:.3f} (p-value = {pearson_p:.4f})")
        print("-" * 60)
        
        color = palette[idx]
        sns.regplot(
            x=lengths, 
            y=thetas, 
            color=color, 
            label=f"{judge_name} ($\\rho \\approx$ {spearman_corr:.2f})",
            scatter_kws={'s': 60, 'alpha': 0.7}, 
            line_kws={'linestyle': '--', 'linewidth': 2}
        )

    plt.title("Verbosity Bias Comparison in LLM Judges: Average Tokens Per Completion vs BTD Score", fontsize=12, fontweight='bold')
    plt.xlabel("Average Completion Tokens per Response", fontsize=10)
    plt.ylabel("BTD Latent Strength ($\\hat{\\theta}$)", fontsize=10)
    plt.legend(loc='upper left', frameon=True)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()
    
    return results_summary

judges_dict = {
    "phi-4": llm_phi4_reactions_scores,
    "qwen3-235b": llm_qwen3_235b_reactions_scores
}

verbosity_results = analyze_verbosity_btd_multiple_judges(
    btd_judges_dict=judges_dict, 
    dataset_path="synthetic_outputs.jsonl"
)